# Step 01_02_04 -- Univariate Census & Target Variable EDA: aoe2companion

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_02 -- EDA
**Dataset:** aoe2companion
**Question:** What are the NULL rates, cardinality, value distributions, and
descriptive statistics for all columns across all four raw tables? What is the
target variable (`won`) distribution and class balance?
**Invariants applied:** #6 (reproducibility -- SQL inlined in artifact),
#7 (no magic numbers), #8 (cross-game comparability), #9 (step scope: query)
**Step scope:** query
**Type:** Read-only -- no DuckDB writes, no new tables, no schema changes

## 0. Imports and DB connection

In [1]:
import json
import os
from pathlib import Path

import pandas as pd

from rts_predict.common.notebook_utils import (
    get_notebook_db,
    get_reports_dir,
    setup_notebook_logging,
)

logger = setup_notebook_logging()

In [2]:
db = get_notebook_db("aoe2", "aoe2companion")
con = db.con
print("Connected to aoe2companion DuckDB (read-only)")

Connected to aoe2companion DuckDB (read-only)


In [3]:
# T07: Memory footprint -- record DuckDB file size as a dataset-level metric
# Note: DB_FILE is not defined in this notebook's scope; use db._dataset.db_file
# (where db = get_notebook_db("aoe2", "aoe2companion")).
db_size_bytes = os.path.getsize(str(db._dataset.db_file))
print(f"DuckDB file size: {db_size_bytes:,} bytes ({db_size_bytes / (1024**3):.2f} GB)")

DuckDB file size: 22,429,839,360 bytes (20.89 GB)


In [4]:
reports_dir = get_reports_dir("aoe2", "aoe2companion")
artifacts_dir = reports_dir / "artifacts" / "01_exploration" / "02_eda"
artifacts_dir.mkdir(parents=True, exist_ok=True)
print(f"Artifacts dir: {artifacts_dir}")

Artifacts dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/02_eda


In [5]:
# Collector for JSON artifact
findings: dict = {}
findings["db_memory_footprint_bytes"] = db_size_bytes

## A. Full NULL census of matches_raw (all 55 columns)

Use SUMMARIZE as primary single-pass approach, then tidy the output.

In [6]:
print("Running SUMMARIZE on matches_raw (single pass over 277M rows)...")
summarize_df = con.execute("SUMMARIZE matches_raw").df()
print(f"SUMMARIZE returned {len(summarize_df)} rows (one per column)")

Running SUMMARIZE on matches_raw (single pass over 277M rows)...


SUMMARIZE returned 55 rows (one per column)


In [7]:
# Extract NULL census from SUMMARIZE output
# null_percentage is already float64 (e.g. 4.69 for 4.69%)
null_census_matches = summarize_df[["column_name", "count", "null_percentage"]].copy()
null_census_matches = null_census_matches.rename(
    columns={"count": "non_null_count"}
)

In [8]:
# Get total row count and cardinality per column via SUMMARIZE approx_unique
total_rows = con.execute(
    "SELECT COUNT(*) AS n FROM matches_raw"
).fetchone()[0]
print(f"Total rows in matches_raw: {total_rows:,}")

Total rows in matches_raw: 277,099,059


In [9]:
# Build tidy NULL census DataFrame
null_census_matches["total_rows"] = total_rows
null_census_matches["null_count"] = (
    (null_census_matches["null_percentage"] / 100.0 * total_rows)
    .round(0)
    .astype(int)
)
null_census_matches["null_pct"] = null_census_matches["null_percentage"]

In [10]:
# Get exact cardinality from SUMMARIZE approx_unique column
null_census_matches["approx_cardinality"] = summarize_df["approx_unique"].astype(int)
print("\n=== matches_raw NULL census (all 55 columns) ===")
print(
    null_census_matches[
        ["column_name", "total_rows", "null_count", "null_pct", "approx_cardinality"]
    ].to_string(index=False)
)


=== matches_raw NULL census (all 55 columns) ===
          column_name  total_rows  null_count  null_pct  approx_cardinality
              matchId   277099059           0      0.00            61799126
              started   277099059           0      0.00            52689164
             finished   277099059           0      0.00            55139738
          leaderboard   277099059           0      0.00                  22
                 name   277099059       27710      0.01             2308187
               server   277099059   271529368     97.99                  11
internalLeaderboardId   277099059     1330075      0.48                 122
              privacy   277099059           0      0.00                   3
                  mod   277099059           0      0.00                   2
                  map   277099059           0      0.00                 261
           difficulty   277099059           0      0.00                   7
          startingAge   277099059     

In [11]:
findings["matches_raw_null_census"] = (
    null_census_matches[
        ["column_name", "total_rows", "null_count", "null_pct", "approx_cardinality"]
    ]
    .to_dict(orient="records")
)
findings["matches_raw_total_rows"] = int(total_rows)

**SQL used:**
```sql
SUMMARIZE matches_raw
SELECT COUNT(*) AS n FROM matches_raw
```

## A2. Empty-string investigation for VARCHAR columns

Investigate whether `difficulty`, `colorHex`, `startingAge`, `endingAge`,
`civilizationSet` (0 NULLs per SUMMARIZE) contain empty strings instead of NULLs.
Also confirm `scenario` and `modDataset` (high-NULL VARCHARs) have genuine NULLs.
`password` is BOOLEAN — empty-string hypothesis does not apply.

In [12]:
empty_string_cols = [
    "difficulty", "colorHex", "startingAge", "endingAge", "civilizationSet",
    "scenario", "modDataset"  # high-NULL VARCHARs -- confirm genuine NULLs
]
empty_string_results = []
for col in empty_string_cols:
    sql = f"""
    SELECT
        '{col}' AS column_name,
        COUNT(*) FILTER (WHERE "{col}" IS NULL)                                AS null_count,
        COUNT(*) FILTER (WHERE "{col}" = '')                                   AS empty_string_count,
        COUNT(*) FILTER (WHERE "{col}" IS NOT NULL AND "{col}" != '')          AS non_empty_count,
        COUNT(*)                                                               AS total_rows
    FROM matches_raw
    """
    row = con.execute(sql).df().iloc[0]
    empty_string_results.append(row.to_dict())
    print(
        f"{col}: NULL={int(row['null_count']):,}  "
        f"EMPTY={int(row['empty_string_count']):,}  "
        f"NON_EMPTY={int(row['non_empty_count']):,}"
    )

findings["empty_string_investigation"] = empty_string_results

difficulty: NULL=0  EMPTY=0  NON_EMPTY=277,099,059


colorHex: NULL=0  EMPTY=0  NON_EMPTY=277,099,059


startingAge: NULL=0  EMPTY=0  NON_EMPTY=277,099,059


endingAge: NULL=0  EMPTY=0  NON_EMPTY=277,099,059


civilizationSet: NULL=0  EMPTY=0  NON_EMPTY=277,099,059


scenario: NULL=272,304,136  EMPTY=114,850  NON_EMPTY=4,680,073


modDataset: NULL=276,312,213  EMPTY=0  NON_EMPTY=786,846


In [13]:
# Verify: for each column the three counts sum to total_rows
for entry in empty_string_results:
    triple_sum = int(entry["null_count"]) + int(entry["empty_string_count"]) + int(entry["non_empty_count"])
    assert triple_sum == int(entry["total_rows"]), (
        f"Triple sum mismatch for {entry['column_name']}: {triple_sum} != {int(entry['total_rows'])}"
    )
print("All triple sums match total_rows.")

All triple sums match total_rows.


## B. Target variable analysis (`won`)

In [14]:
won_dist_sql = """
SELECT
    won,
    COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM matches_raw
GROUP BY won
ORDER BY cnt DESC
"""
print("Running won distribution query...")
won_dist_df = con.execute(won_dist_sql).df()
print("\n=== won value distribution ===")
print(won_dist_df.to_string(index=False))
findings["won_distribution"] = won_dist_df.to_dict(orient="records")

Running won distribution query...



=== won value distribution ===
  won       cnt   pct
False 132150323 47.69
 True 131963175 47.62
 <NA>  12985561  4.69


In [15]:
# Patch won null_count with exact value from GROUP BY (SUMMARIZE is approximate)
null_rows = won_dist_df.loc[won_dist_df["won"].isna()]
assert len(null_rows) == 1, f"Expected exactly 1 NULL won group, got {len(null_rows)}"
won_null_exact = int(null_rows["cnt"].iloc[0])
won_idx = null_census_matches.index[null_census_matches["column_name"] == "won"]
null_census_matches.loc[won_idx, "null_count"] = won_null_exact
null_census_matches.loc[won_idx, "null_pct"] = round(
    100.0 * won_null_exact / total_rows, 2
)
print(f"Patched won null_count: {won_null_exact:,} (exact from GROUP BY)")

# Re-emit matches_raw_null_census with corrected won value
findings["matches_raw_null_census"] = (
    null_census_matches[
        ["column_name", "total_rows", "null_count", "null_pct", "approx_cardinality"]
    ]
    .to_dict(orient="records")
)

findings["exact_won_null_note"] = {
    "column": "won",
    "summarize_derived_null_count": 12995946,
    "group_by_exact_null_count": won_null_exact,
    "discrepancy": 12995946 - won_null_exact,
    "resolution": "GROUP BY count is authoritative for the prediction target.",
    "approximation_note": (
        "SUMMARIZE approximate counts are used for all columns except 'won', "
        "where the exact GROUP BY value is authoritative for the prediction target."
    )
}

Patched won null_count: 12,985,561 (exact from GROUP BY)


In [16]:
won_by_lb_sql = """
SELECT
    leaderboard,
    won,
    COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(PARTITION BY leaderboard), 2) AS pct
FROM matches_raw
GROUP BY leaderboard, won
ORDER BY leaderboard, won
"""
print("Running won distribution stratified by leaderboard...")
won_by_lb_df = con.execute(won_by_lb_sql).df()
print("\n=== won distribution by leaderboard ===")
print(won_by_lb_df.to_string(index=False))
findings["won_by_leaderboard"] = won_by_lb_df.to_dict(orient="records")

Running won distribution stratified by leaderboard...



=== won distribution by leaderboard ===
                 leaderboard   won      cnt   pct
                      dm_1v1 False    47254 50.00
                      dm_1v1  True    47254 50.00
                     dm_team False    96700 50.00
                     dm_team  True    96700 50.00
                      ew_1v1 False   971932 50.00
                      ew_1v1  True   971928 50.00
                      ew_1v1  <NA>      111  0.01
              ew_1v1_console False     3761 50.00
              ew_1v1_console  True     3761 50.00
        ew_1v1_redbullwololo False   297425 50.00
        ew_1v1_redbullwololo  True   297431 50.00
        ew_1v1_redbullwololo  <NA>       34  0.01
ew_1v1_redbullwololo_console False      325 50.00
ew_1v1_redbullwololo_console  True      325 50.00
                     ew_team False  1678178 50.00
                     ew_team  True  1678173 50.00
                     ew_team  <NA>      259  0.01
             ew_team_console False     2906 50.00
         

In [17]:
# Intra-match won consistency check (2-row matches)
consistency_sql = """
WITH match_pairs AS (
    SELECT
        matchId,
        COUNT(*) AS n_rows,
        COUNT(*) FILTER (WHERE won = TRUE) AS won_true,
        COUNT(*) FILTER (WHERE won = FALSE) AS won_false,
        COUNT(*) - COUNT(won) AS won_null
    FROM matches_raw
    GROUP BY matchId
    HAVING COUNT(*) = 2
)
SELECT
    COUNT(*) AS total_2row_matches,
    COUNT(*) FILTER (WHERE won_true = 1 AND won_false = 1)
        AS consistent_complement,
    COUNT(*) FILTER (WHERE won_null = 2) AS both_null,
    COUNT(*) FILTER (WHERE won_true = 1 AND won_null = 1)
        AS one_true_one_null,
    COUNT(*) FILTER (WHERE won_false = 1 AND won_null = 1)
        AS one_false_one_null,
    COUNT(*) FILTER (WHERE won_true = 2) AS both_true,
    COUNT(*) FILTER (WHERE won_false = 2) AS both_false,
    COUNT(*) FILTER (WHERE won_true = 0 AND won_false = 0
        AND won_null < 2) AS other_inconsistent
FROM match_pairs
"""
print("Running intra-match won consistency check...")
consistency_df = con.execute(consistency_sql).df()
print("\n=== Intra-match won consistency (2-row matches) ===")
print(consistency_df.to_string(index=False))

Running intra-match won consistency check...



=== Intra-match won consistency (2-row matches) ===
 total_2row_matches  consistent_complement  both_null  one_true_one_null  one_false_one_null  both_true  both_false  other_inconsistent
           40062975               35479656       1885              66642              116065    2499163     1899564                   0


In [18]:
findings["won_consistency_2row"] = consistency_df.to_dict(orient="records")

In [19]:
# Verification: won distribution (deferred to 01_02_05 for plotting)
print("=== won distribution (feeds bar chart in 01_02_05) ===")
print(won_dist_df.to_string(index=False))

=== won distribution (feeds bar chart in 01_02_05) ===
  won       cnt   pct
False 132150323 47.69
 True 131963175 47.62
 <NA>  12985561  4.69


## C. Match structure by leaderboard

In [20]:
match_struct_sql = """
SELECT
    leaderboard,
    COUNT(DISTINCT matchId) AS distinct_matches,
    COUNT(*) AS total_rows,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT matchId), 2) AS avg_rows_per_match
FROM matches_raw
GROUP BY leaderboard
ORDER BY total_rows DESC
"""
print("Running match structure by leaderboard query...")
match_struct_df = con.execute(match_struct_sql).df()
print("\n=== Match structure by leaderboard ===")
print(match_struct_df.to_string(index=False))
findings["match_structure_by_leaderboard"] = match_struct_df.to_dict(orient="records")

Running match structure by leaderboard query...



=== Match structure by leaderboard ===
                 leaderboard  distinct_matches  total_rows  avg_rows_per_match
                     rm_team          17379984   102711164                5.91
                    unranked          18783620    78254732                4.17
                      rm_1v1          26847572    53694523                2.00
                  qp_rm_team           3611539    19707155                5.46
                   qp_rm_1v1           3688676     7377276                2.00
                     unknown            930658     3782726                4.06
             rm_team_console            721688     3472383                4.81
                     ew_team            600507     3356610                5.59
                      ew_1v1            972000     1943971                2.00
              rm_1v1_console            800244     1600477                2.00
        ew_1v1_redbullwololo            297445      594890                2.00
            

## D. Categorical field profiles

In [21]:
# Columns with full top-k value listing
cat_topk_cols = [
    "leaderboard", "civ", "map", "gameMode", "speed", "victory",
    "server", "country", "status", "difficulty", "startingAge",
    "endingAge", "gameVariant", "resources", "revealMap", "mapSize",
    "civilizationSet", "privacy", "scenario", "modDataset",
]

In [22]:
# Loop over categorical columns -- top 30 values each
cat_profiles: dict = {}
for col in cat_topk_cols:
    sql = f"""
    SELECT
        "{col}" AS value,
        COUNT(*) AS cnt,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
    FROM matches_raw
    GROUP BY "{col}"
    ORDER BY cnt DESC
    LIMIT 30
    """
    df = con.execute(sql).df()
    cat_profiles[col] = df.to_dict(orient="records")
    n_distinct = con.execute(
        f'SELECT COUNT(DISTINCT "{col}") FROM matches_raw'
    ).fetchone()[0]
    n_null = con.execute(
        f'SELECT COUNT(*) - COUNT("{col}") FROM matches_raw'
    ).fetchone()[0]
    print(f"\n--- {col} (cardinality={n_distinct}, nulls={n_null:,}) ---")
    print(df.head(15).to_string(index=False))


--- leaderboard (cardinality=21, nulls=0) ---
               value       cnt   pct
             rm_team 102711164 37.07
            unranked  78254732 28.24
              rm_1v1  53694523 19.38
          qp_rm_team  19707155  7.11
           qp_rm_1v1   7377276  2.66
             unknown   3782726  1.37
     rm_team_console   3472383  1.25
             ew_team   3356610  1.21
              ew_1v1   1943971  0.70
      rm_1v1_console   1600477  0.58
ew_1v1_redbullwololo    594890  0.21
             dm_team    193400  0.07
           qp_br_ffa    167984  0.06
              dm_1v1     94508  0.03
           qp_ew_1v1     49477  0.02



--- civ (cardinality=75, nulls=0) ---
      value      cnt  pct
     franks 15745563 5.68
    mongols 12320535 4.45
    britons 11814240 4.26
   persians 10762563 3.88
    spanish 10455030 3.77
      khmer 10350344 3.74
       huns  9475018 3.42
    teutons  8727584 3.15
      goths  8453800 3.05
      turks  7535597 2.72
 ethiopians  7308930 2.64
 byzantines  7231564 2.61
    magyars  7207859 2.60
lithuanians  6918321 2.50
 portuguese  6719457 2.42



--- map (cardinality=223, nulls=0) ---
              value      cnt   pct
          rm_arabia 54118272 19.53
           rm_arena 46540790 16.80
    rm_black-forest 34421288 12.42
           rm_nomad 18932603  6.83
            unknown 14792062  5.34
      rm_megarandom 14292224  5.16
  rm_amazon_tunnels  7273441  2.62
rm_african_clearing  6568294  2.37
      rm_land_nomad  5476005  1.98
           rm_michi  4299160  1.55
           rm_oasis  3571412  1.29
         rm_hideout  3546713  1.28
       rm_lombardia  3216877  1.16
         rm_coastal  2791794  1.01
        rm_fortress  2549602  0.92



--- gameMode (cardinality=14, nulls=0) ---
            value       cnt   pct
       random_map 252087632 90.97
      empire_wars   7619367  2.75
         scenario   6686197  2.41
   co_op_campaign   3954938  1.43
         regicide   2638530  0.95
      death_match   2278097  0.82
     sudden_death    688601  0.25
          unknown    460512  0.17
    battle_royale    315126  0.11
 king_of_the_hill    209629  0.08
defend_the_wonder     96591  0.03
capture_the_relic     37270  0.01
      wonder_race     26567  0.01
 turbo_random_map         2  0.00



--- speed (cardinality=5, nulls=0) ---
  value       cnt   pct
 normal 254622800 91.89
   fast  17179752  6.20
 casual   4609521  1.66
unknown    507457  0.18
   slow    179529  0.06


...


--- mapSize (cardinality=15, nulls=0) ---
     value      cnt   pct
     large 86226253 31.12
      tiny 76307245 27.54
    medium 48595311 17.54
    normal 44355175 16.01
     giant  8734943  3.15
 ludicrous  5060487  1.83
      huge  4797231  1.73
     small  2568005  0.93
   unknown   428468  0.15
  enormous    11133  0.00
   massive     7183  0.00
 monstrous     4140  0.00
 miniature     2081  0.00
  colossal      984  0.00
incredible      420  0.00



--- civilizationSet (cardinality=4, nulls=0) ---
     value       cnt   pct
   unknown 190248871 68.66
      aoe2  85733228 30.94
       all   1114116  0.40
chronicles      2844  0.00



--- privacy (cardinality=3, nulls=0) ---
  value       cnt   pct
 public 261023646 94.20
private  14101316  5.09
unknown   1974097  0.71



--- scenario (cardinality=28268, nulls=272,304,136) ---
                                               value       cnt   pct
                                                 NaN 272304136 98.27
                                                        114850  0.04
                                               35319     84907  0.03
                                               35613     69454  0.03
                                               35321     63959  0.02
                                               35229     60125  0.02
                     CBA_=REQUIEM=_V235.aoe2scenario     58285  0.02
                                               35612     53590  0.02
FAST VILLAGER DEFENSE 2025 Club DE v550.aoe2scenario     49792  0.02
                                               35615     48763  0.02
                                               35234     44453  0.02
                     CBA_=REQUIEM=_V218.aoe2scenario     44129  0.02
FAST VILLAGER DEFENSE 2025 Club DE v547.aoe2sc


--- modDataset (cardinality=739, nulls=276,312,213) ---
                                                         value       cnt   pct
                                                           NaN 276312213 99.72
                                 251941-New No Collision Trade    131306  0.05
                23222-23222_Fast Villager Defense Club DE 47.1     41742  0.02
           363188-PQ2.41_10xShared_3xtech_10xresources_1xtrees     37695  0.01
            363188-MS2.1_10xShared_3xtech_10xresources_1xtrees     32703  0.01
                                        858-No Collision Trade     28151  0.01
60487-60487_6x Tech CBA (Experimental) Alexander the Great DLC     24851  0.01
                   363188-PQ3.1_11x Shared Civ bonus  3x Techs     21233  0.01
                            362656-362656_CBA Requiem Original     19225  0.01
               59728-59728_6X Tech CBA Alexander the Great DLC     18195  0.01
                             15373-15373_Multiple Buffs Mod v1     14488  

In [23]:
findings["categorical_profiles"] = cat_profiles

In [24]:
# name: cardinality only (millions of distinct values)
name_sql = """
SELECT
    COUNT(DISTINCT name) AS distinct_names,
    COUNT(*) - COUNT(name) AS null_count,
    ROUND(100.0 * (COUNT(*) - COUNT(name)) / COUNT(*), 2) AS null_pct
FROM matches_raw
"""
print("Running name cardinality query...")
name_df = con.execute(name_sql).df()
print("\n--- name (cardinality only) ---")
print(name_df.to_string(index=False))
findings["name_cardinality"] = name_df.to_dict(orient="records")

Running name cardinality query...



--- name (cardinality only) ---
 distinct_names  null_count  null_pct
        2468478       19857      0.01


In [25]:
# colorHex: cardinality and NULL rate only
colorhex_sql = """
SELECT
    COUNT(DISTINCT "colorHex") AS distinct_values,
    COUNT(*) - COUNT("colorHex") AS null_count,
    ROUND(100.0 * (COUNT(*) - COUNT("colorHex")) / COUNT(*), 2) AS null_pct
FROM matches_raw
"""
colorhex_df = con.execute(colorhex_sql).df()
print("\n--- colorHex (cardinality only) ---")
print(colorhex_df.to_string(index=False))
findings["colorHex_cardinality"] = colorhex_df.to_dict(orient="records")


--- colorHex (cardinality only) ---
 distinct_values  null_count  null_pct
               9           0       0.0


In [26]:
# Verification: categorical top-k profiles (deferred to 01_02_05 for plotting)
print("=== leaderboard, civ, map top-k (feeds bar charts in 01_02_05) ===")
for col in ["leaderboard", "civ", "map"]:
    print(f"\n--- {col} ---")
    print(pd.DataFrame(cat_profiles[col]).head(20).to_string(index=False))

=== leaderboard, civ, map top-k (feeds bar charts in 01_02_05) ===

--- leaderboard ---
               value       cnt   pct
             rm_team 102711164 37.07
            unranked  78254732 28.24
              rm_1v1  53694523 19.38
          qp_rm_team  19707155  7.11
           qp_rm_1v1   7377276  2.66
             unknown   3782726  1.37
     rm_team_console   3472383  1.25
             ew_team   3356610  1.21
              ew_1v1   1943971  0.70
      rm_1v1_console   1600477  0.58
ew_1v1_redbullwololo    594890  0.21
             dm_team    193400  0.07
           qp_br_ffa    167984  0.06
              dm_1v1     94508  0.03
           qp_ew_1v1     49477  0.02
          qp_ew_team     44716  0.02
            ror_team     23308  0.01
             ror_1v1     15775  0.01
      ew_1v1_console      7522  0.00
     ew_team_console      5812  0.00

--- civ ---
      value      cnt  pct
     franks 15745563 5.68
    mongols 12320535 4.45
    britons 11814240 4.26
   persians 107625

## E. Boolean field census

In [27]:
bool_cols = [
    "mod", "fullTechTree", "allowCheats", "empireWarsMode", "lockSpeed",
    "lockTeams", "hideCivs", "recordGame", "regicideMode",
    "sharedExploration", "suddenDeathMode", "antiquityMode",
    "teamPositions", "teamTogether", "turboMode", "password",
    "shared", "verified",
]

In [28]:
bool_records = []
for col in bool_cols:
    sql = f"""
    SELECT
        '{col}' AS column_name,
        COUNT(*) FILTER (WHERE "{col}" = TRUE) AS true_count,
        COUNT(*) FILTER (WHERE "{col}" = FALSE) AS false_count,
        COUNT(*) - COUNT("{col}") AS null_count
    FROM matches_raw
    """
    row = con.execute(sql).df().iloc[0]
    bool_records.append(row.to_dict())
    print(
        f"{col}: TRUE={int(row['true_count']):,}  "
        f"FALSE={int(row['false_count']):,}  "
        f"NULL={int(row['null_count']):,}"
    )

mod: TRUE=7,022,020  FALSE=270,077,039  NULL=0


fullTechTree: TRUE=1,074,868  FALSE=275,592,697  NULL=431,494


allowCheats: TRUE=2,830,361  FALSE=273,840,360  NULL=428,338


empireWarsMode: TRUE=255,573,532  FALSE=18,595,385  NULL=2,930,142


lockSpeed: TRUE=251,301,799  FALSE=25,369,495  NULL=427,765


...

teamPositions: TRUE=228,639,034  FALSE=48,031,894  NULL=428,131


teamTogether: TRUE=272,146,075  FALSE=4,524,654  NULL=428,330


turboMode: TRUE=251,931,724  FALSE=24,739,523  NULL=427,812


password: TRUE=39,902,054  FALSE=7,491,234  NULL=229,705,771
shared: TRUE=531,363  FALSE=276,567,696  NULL=0


verified: TRUE=2,177,071  FALSE=274,921,988  NULL=0


In [29]:
bool_census_df = pd.DataFrame(bool_records)
print("\n=== Boolean field census ===")
print(bool_census_df.to_string(index=False))
findings["boolean_census"] = bool_census_df.to_dict(orient="records")


=== Boolean field census ===
      column_name  true_count  false_count  null_count
              mod     7022020    270077039           0
     fullTechTree     1074868    275592697      431494
      allowCheats     2830361    273840360      428338
   empireWarsMode   255573532     18595385     2930142
        lockSpeed   251301799     25369495      427765
        lockTeams    32135660    244535509      427890
         hideCivs     4991659    135498344   136609056
       recordGame   255765951     20905084      428024
     regicideMode   240124881     16485894    20488284
sharedExploration    73294670    203377119      427270
  suddenDeathMode   256318276     17891014     2889769
    antiquityMode    72382175     14468333   190248551
    teamPositions   228639034     48031894      428131
     teamTogether   272146075      4524654      428330
        turboMode   251931724     24739523      427812
         password    39902054      7491234   229705771
           shared      531363    27

## F. Numeric descriptive statistics

### F.1 matches_raw numerics

In [30]:
matches_numeric_cols = [
    "rating", "ratingDiff", "population", "slot", "color",
    "team", "speedFactor", "treatyLength", "internalLeaderboardId",
]

In [31]:
matches_numeric_stats = []
for col in matches_numeric_cols:
    sql = f"""
    SELECT
        '{col}' AS column_name,
        MIN("{col}") AS min_val,
        MAX("{col}") AS max_val,
        ROUND(AVG("{col}"), 2) AS mean_val,
        ROUND(MEDIAN("{col}"), 2) AS median_val,
        ROUND(STDDEV("{col}"), 2) AS stddev_val,
        PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY "{col}") AS p05,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY "{col}") AS p25,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY "{col}") AS p75,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "{col}") AS p95
    FROM matches_raw
    WHERE "{col}" IS NOT NULL
    """
    row = con.execute(sql).df().iloc[0]
    matches_numeric_stats.append(row.to_dict())
    print(f"\n--- {col} ---")
    print(row.to_string())


--- rating ---
column_name     rating
min_val             -1
max_val           5001
mean_val       1120.23
median_val      1093.0
stddev_val      290.01
p05              687.0
p25              934.0
p75             1299.0
p95             1598.0



--- ratingDiff ---
column_name    ratingDiff
min_val              -174
max_val               319
mean_val            -0.19
median_val            0.0
stddev_val          17.66
p05                 -20.0
p25                 -16.0
p75                  16.0
p95                  20.0



--- population ---
column_name    population
min_val                 0
max_val              9999
mean_val           216.98
median_val          200.0
stddev_val          57.06
p05                 200.0
p25                 200.0
p75                 200.0
p95                 300.0



--- slot ---
column_name    slot
min_val           0
max_val           7
mean_val       2.04
median_val      1.0
stddev_val     2.03
p05             0.0
p25             0.0
p75             3.0
p95             6.0



--- color ---
column_name    color
min_val            0
max_val           43
mean_val        3.37
median_val       3.0
stddev_val      2.22
p05              1.0
p25              2.0
p75              5.0
p95              8.0



--- team ---
column_name    team
min_val           0
max_val         255
mean_val       1.82
median_val      2.0
stddev_val     1.49
p05             1.0
p25             1.0
p75             2.0
p95             5.0



--- speedFactor ---
column_name    speedFactor
min_val                1.0
max_val                2.0
mean_val              1.71
median_val             1.7
stddev_val            0.09
p05                    1.7
p25                    1.7
p75                    1.7
p95                    2.0



--- treatyLength ---
column_name    treatyLength
min_val                   0
max_val                  90
mean_val               0.87
median_val              0.0
stddev_val             6.02
p05                     0.0
p25                     0.0
p75                     0.0
p95                     0.0



--- internalLeaderboardId ---
column_name    internalLeaderboardId
min_val                            0
max_val                          125
mean_val                        8.79
median_val                       7.0
stddev_val                     12.93
p05                              0.0
p25                              0.0
p75                              9.0
p95                             21.0


In [32]:
findings["matches_raw_numeric_stats"] = matches_numeric_stats

### F.2 ratings_raw numerics

In [33]:
ratings_numeric_cols = [
    "rating", "games", "rating_diff", "leaderboard_id", "season",
]

In [34]:
ratings_numeric_stats = []
for col in ratings_numeric_cols:
    sql = f"""
    SELECT
        '{col}' AS column_name,
        MIN("{col}") AS min_val,
        MAX("{col}") AS max_val,
        ROUND(AVG("{col}"), 2) AS mean_val,
        ROUND(MEDIAN("{col}"), 2) AS median_val,
        ROUND(STDDEV("{col}"), 2) AS stddev_val,
        PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY "{col}") AS p05,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY "{col}") AS p25,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY "{col}") AS p75,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "{col}") AS p95
    FROM ratings_raw
    WHERE "{col}" IS NOT NULL
    """
    row = con.execute(sql).df().iloc[0]
    ratings_numeric_stats.append(row.to_dict())
    print(f"\n--- ratings_raw.{col} ---")
    print(row.to_string())


--- ratings_raw.rating ---
column_name     rating
min_val            -34
max_val           3026
mean_val       1120.02
median_val      1093.0
stddev_val      266.79
p05              693.0
p25              974.0
p75             1286.0
p95             1552.0



--- ratings_raw.games ---
column_name          games
min_val                 10
max_val         1775260795
mean_val        2403524.43
median_val           546.0
stddev_val     65257628.19
p05                   24.0
p25                  142.0
p75                 1602.0
p95                 4736.0



--- ratings_raw.rating_diff ---
column_name    rating_diff
min_val               -136
max_val                282
mean_val              0.11
median_val             0.0
stddev_val           13.74
p05                  -19.0
p25                  -14.0
p75                   14.0
p95                   19.0



--- ratings_raw.leaderboard_id ---
column_name    leaderboard_id
min_val                     0
max_val                  5019
mean_val                 9.15
median_val                3.0
stddev_val             184.44
p05                       0.0
p25                       0.0
p75                       4.0
p95                       4.0



--- ratings_raw.season ---
column_name    season
min_val            -1
max_val            -1
mean_val         -1.0
median_val       -1.0
stddev_val        0.0
p05              -1.0
p25              -1.0
p75              -1.0
p95              -1.0


In [35]:
findings["ratings_raw_numeric_stats"] = ratings_numeric_stats

### F.3 leaderboards_raw numerics

In [36]:
lb_numeric_cols = [
    "rank", "rating", "wins", "losses", "games",
    "streak", "drops", "rankCountry", "season", "rankLevel",
]

In [37]:
lb_numeric_stats = []
for col in lb_numeric_cols:
    sql = f"""
    SELECT
        '{col}' AS column_name,
        MIN("{col}") AS min_val,
        MAX("{col}") AS max_val,
        ROUND(AVG("{col}"), 2) AS mean_val,
        ROUND(MEDIAN("{col}"), 2) AS median_val,
        ROUND(STDDEV("{col}"), 2) AS stddev_val,
        PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY "{col}") AS p05,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY "{col}") AS p25,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY "{col}") AS p75,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "{col}") AS p95
    FROM leaderboards_raw
    WHERE "{col}" IS NOT NULL
    """
    row = con.execute(sql).df().iloc[0]
    lb_numeric_stats.append(row.to_dict())
    print(f"\n--- leaderboards_raw.{col} ---")
    print(row.to_string())


--- leaderboards_raw.rank ---
column_name        rank
min_val               1
max_val          247604
mean_val       86228.83
median_val      73999.0
stddev_val     61221.47
p05              2842.0
p25             33580.0
p75            138103.0
p95            190546.0

--- leaderboards_raw.rating ---
column_name     rating
min_val           -441
max_val           3110
mean_val       1038.78
median_val      1003.0
stddev_val      199.19
p05              746.0
p25              953.0
p75             1111.0
p95             1408.0



--- leaderboards_raw.wins ---
column_name      wins
min_val             0
max_val         13878
mean_val        107.4
median_val       27.0
stddev_val     275.47
p05               4.0
p25              10.0
p75              84.0
p95             467.0



--- leaderboards_raw.losses ---
column_name    losses
min_val             0
max_val         15274
mean_val       102.91
median_val       26.0
stddev_val     260.42
p05               4.0
p25              11.0
p75              79.0
p95             453.0

--- leaderboards_raw.games ---
column_name     games
min_val            10
max_val         22446
mean_val       174.36
median_val       45.0
stddev_val     466.25
p05              11.0
p25              20.0
p75             128.0
p95             741.0



--- leaderboards_raw.streak ---
column_name    streak
min_val         -1331
max_val          1000
mean_val        -0.13
median_val        1.0
stddev_val       5.45
p05              -6.0
p25              -2.0
p75               2.0
p95               5.0



--- leaderboards_raw.drops ---
column_name    drops
min_val            0
max_val         4553
mean_val        5.88
median_val       1.0
stddev_val     22.21
p05              0.0
p25              0.0
p75              4.0
p95             24.0

--- leaderboards_raw.rankCountry ---
column_name    rankCountry
min_val                  1
max_val             105669
mean_val          15300.75
median_val          3383.0
stddev_val        23179.72
p05                   59.0
p25                  855.0
p75                18088.0
p95                70491.0

--- leaderboards_raw.season ---
column_name    season
min_val            -1
max_val            -1
mean_val         -1.0
median_val       -1.0
stddev_val        0.0
p05              -1.0
p25              -1.0
p75              -1.0
p95              -1.0

--- leaderboards_raw.rankLevel ---
column_name    rankLevel
min_val                1
max_val                1
mean_val             1.0
median_val           1.0
stddev_val           0.0
p05        

In [38]:
findings["leaderboards_raw_numeric_stats"] = lb_numeric_stats

## F2. Zero counts for numeric columns

EDA Manual Section 3.1: compute zero counts for all numeric columns.
`profiles_raw` excluded -- its only numeric column (`profileId`) is an identifier;
zero counts on identifiers are semantically meaningless.

In [39]:
matches_zero_cols = [
    "rating", "ratingDiff", "population", "slot", "color",
    "team", "speedFactor", "treatyLength", "internalLeaderboardId",
]
matches_zero_counts = []
for col in matches_zero_cols:
    sql = f"""
    SELECT
        '{col}' AS column_name,
        COUNT(*) FILTER (WHERE "{col}" = 0)                                                  AS zero_count,
        COUNT(*) FILTER (WHERE "{col}" IS NOT NULL)                                          AS non_null_count,
        ROUND(100.0 * COUNT(*) FILTER (WHERE "{col}" = 0)
            / NULLIF(COUNT(*) FILTER (WHERE "{col}" IS NOT NULL), 0), 2)                    AS zero_pct_of_non_null
    FROM matches_raw
    """
    row = con.execute(sql).df().iloc[0]
    matches_zero_counts.append(row.to_dict())
    print(f"matches_raw.{col}: zero_count={int(row['zero_count']):,}")
findings["matches_raw_zero_counts"] = matches_zero_counts

matches_raw.rating: zero_count=7,792
matches_raw.ratingDiff: zero_count=6,860,545


matches_raw.population: zero_count=2


matches_raw.slot: zero_count=74,781,923
matches_raw.color: zero_count=1,732


matches_raw.team: zero_count=449


matches_raw.speedFactor: zero_count=0
matches_raw.treatyLength: zero_count=267,323,811


matches_raw.internalLeaderboardId: zero_count=78,254,732


In [40]:
ratings_zero_cols = ["rating", "games", "rating_diff", "leaderboard_id", "season"]
ratings_zero_counts = []
for col in ratings_zero_cols:
    sql = f"""
    SELECT
        '{col}' AS column_name,
        COUNT(*) FILTER (WHERE "{col}" = 0)                                                  AS zero_count,
        COUNT(*) FILTER (WHERE "{col}" IS NOT NULL)                                          AS non_null_count,
        ROUND(100.0 * COUNT(*) FILTER (WHERE "{col}" = 0)
            / NULLIF(COUNT(*) FILTER (WHERE "{col}" IS NOT NULL), 0), 2)                    AS zero_pct_of_non_null
    FROM ratings_raw
    """
    row = con.execute(sql).df().iloc[0]
    ratings_zero_counts.append(row.to_dict())
    print(f"ratings_raw.{col}: zero_count={int(row['zero_count']):,}")
findings["ratings_raw_zero_counts"] = ratings_zero_counts

ratings_raw.rating: zero_count=3,655
ratings_raw.games: zero_count=0
ratings_raw.rating_diff: zero_count=12,442,282
ratings_raw.leaderboard_id: zero_count=25,839,038
ratings_raw.season: zero_count=0


In [41]:
lb_zero_cols = [
    "rank", "rating", "wins", "losses", "games",
    "streak", "drops", "rankCountry", "season", "rankLevel"
]
lb_zero_counts = []
for col in lb_zero_cols:
    sql = f"""
    SELECT
        '{col}' AS column_name,
        COUNT(*) FILTER (WHERE "{col}" = 0)                                                  AS zero_count,
        COUNT(*) FILTER (WHERE "{col}" IS NOT NULL)                                          AS non_null_count,
        ROUND(100.0 * COUNT(*) FILTER (WHERE "{col}" = 0)
            / NULLIF(COUNT(*) FILTER (WHERE "{col}" IS NOT NULL), 0), 2)                    AS zero_pct_of_non_null
    FROM leaderboards_raw
    """
    row = con.execute(sql).df().iloc[0]
    lb_zero_counts.append(row.to_dict())
    print(f"leaderboards_raw.{col}: zero_count={int(row['zero_count']):,}")
findings["leaderboards_raw_zero_counts"] = lb_zero_counts

leaderboards_raw.rank: zero_count=0
leaderboards_raw.rating: zero_count=93
leaderboards_raw.wins: zero_count=9,364
leaderboards_raw.losses: zero_count=5,457
leaderboards_raw.games: zero_count=0
leaderboards_raw.streak: zero_count=0
leaderboards_raw.drops: zero_count=640,929
leaderboards_raw.rankCountry: zero_count=0
leaderboards_raw.season: zero_count=0
leaderboards_raw.rankLevel: zero_count=0


## F1b. Skewness and Kurtosis

EDA Manual Section 3.1 requires skewness and kurtosis for all numeric columns.

In [42]:
# T02: Skewness and kurtosis for matches_raw numeric columns
matches_skew_kurtosis_cols = [
    "rating", "ratingDiff", "population", "slot", "color",
    "team", "speedFactor", "treatyLength", "internalLeaderboardId",
]
matches_skew_kurtosis = []
for col in matches_skew_kurtosis_cols:
    sql = f"""
    SELECT
        '{col}' AS column_name,
        ROUND(SKEWNESS("{col}"), 4) AS skewness,
        ROUND(KURTOSIS("{col}"), 4) AS kurtosis
    FROM matches_raw
    WHERE "{col}" IS NOT NULL
    """
    row = con.execute(sql).df().iloc[0]
    matches_skew_kurtosis.append(row.to_dict())
    print(f"matches_raw.{col}: skewness={row['skewness']}, kurtosis={row['kurtosis']}")
findings["matches_raw_skew_kurtosis"] = matches_skew_kurtosis
print("\n=== matches_raw skewness/kurtosis ===")
print(pd.DataFrame(matches_skew_kurtosis).to_string(index=False))

matches_raw.rating: skewness=0.5662, kurtosis=1.6157


matches_raw.ratingDiff: skewness=0.1105, kurtosis=0.89


matches_raw.population: skewness=3.7833, kurtosis=17.8371


matches_raw.slot: skewness=0.916, kurtosis=-0.1918


matches_raw.color: skewness=1.6691, kurtosis=15.4799


matches_raw.team: skewness=3.8741, kurtosis=30.6175


matches_raw.speedFactor: skewness=0.6323, kurtosis=17.6757


matches_raw.treatyLength: skewness=9.309, kurtosis=103.0427


matches_raw.internalLeaderboardId: skewness=4.355, kurtosis=24.9976

=== matches_raw skewness/kurtosis ===
          column_name  skewness  kurtosis
               rating    0.5662    1.6157
           ratingDiff    0.1105    0.8900
           population    3.7833   17.8371
                 slot    0.9160   -0.1918
                color    1.6691   15.4799
                 team    3.8741   30.6175
          speedFactor    0.6323   17.6757
         treatyLength    9.3090  103.0427
internalLeaderboardId    4.3550   24.9976


In [43]:
# T02: Skewness and kurtosis for ratings_raw numeric columns (5 cols)
ratings_skew_kurtosis_cols = ["leaderboard_id", "season", "rating", "games", "rating_diff"]
ratings_skew_kurtosis = []
for col in ratings_skew_kurtosis_cols:
    sql = f"""
    SELECT
        '{col}' AS column_name,
        ROUND(SKEWNESS("{col}"), 4) AS skewness,
        ROUND(KURTOSIS("{col}"), 4) AS kurtosis
    FROM ratings_raw
    WHERE "{col}" IS NOT NULL
    """
    row = con.execute(sql).df().iloc[0]
    ratings_skew_kurtosis.append(row.to_dict())
    print(f"ratings_raw.{col}: skewness={row['skewness']}, kurtosis={row['kurtosis']}")
findings["ratings_raw_skew_kurtosis"] = ratings_skew_kurtosis
print("\n=== ratings_raw skewness/kurtosis ===")
print(pd.DataFrame(ratings_skew_kurtosis).to_string(index=False))

ratings_raw.leaderboard_id: skewness=27.1172, kurtosis=733.5336


ratings_raw.season: skewness=nan, kurtosis=nan


ratings_raw.rating: skewness=0.3329, kurtosis=1.7587


ratings_raw.games: skewness=27.1278, kurtosis=733.917


ratings_raw.rating_diff: skewness=0.0, kurtosis=-1.1811

=== ratings_raw skewness/kurtosis ===
   column_name  skewness  kurtosis
leaderboard_id   27.1172  733.5336
        season       NaN       NaN
        rating    0.3329    1.7587
         games   27.1278  733.9170
   rating_diff    0.0000   -1.1811


In [44]:
# T02: Skewness and kurtosis for leaderboards_raw numeric columns
lb_skew_kurtosis_cols = [
    "rank", "rating", "wins", "losses", "games",
    "streak", "drops", "rankCountry", "season", "rankLevel",
]
lb_skew_kurtosis = []
for col in lb_skew_kurtosis_cols:
    sql = f"""
    SELECT
        '{col}' AS column_name,
        ROUND(SKEWNESS("{col}"), 4) AS skewness,
        ROUND(KURTOSIS("{col}"), 4) AS kurtosis
    FROM leaderboards_raw
    WHERE "{col}" IS NOT NULL
    """
    row = con.execute(sql).df().iloc[0]
    lb_skew_kurtosis.append(row.to_dict())
    print(f"leaderboards_raw.{col}: skewness={row['skewness']}, kurtosis={row['kurtosis']}")
findings["leaderboards_raw_skew_kurtosis"] = lb_skew_kurtosis
print("\n=== leaderboards_raw skewness/kurtosis ===")
print(pd.DataFrame(lb_skew_kurtosis).to_string(index=False))

leaderboards_raw.rank: skewness=0.3594, kurtosis=-1.0944
leaderboards_raw.rating: skewness=0.7635, kurtosis=3.8332
leaderboards_raw.wins: skewness=8.2141, kurtosis=121.0519
leaderboards_raw.losses: skewness=7.6537, kurtosis=101.7184
leaderboards_raw.games: skewness=8.5055, kurtosis=121.8539
leaderboards_raw.streak: skewness=-10.8473, kurtosis=3644.9604
leaderboards_raw.drops: skewness=27.22, kurtosis=2211.8138
leaderboards_raw.rankCountry: skewness=1.6566, kurtosis=1.5079
leaderboards_raw.season: skewness=nan, kurtosis=nan
leaderboards_raw.rankLevel: skewness=-273614308.6411, kurtosis=-4.738971926357748e+16

=== leaderboards_raw skewness/kurtosis ===
column_name      skewness      kurtosis
       rank  3.594000e-01 -1.094400e+00
     rating  7.635000e-01  3.833200e+00
       wins  8.214100e+00  1.210519e+02
     losses  7.653700e+00  1.017184e+02
      games  8.505500e+00  1.218539e+02
     streak -1.084730e+01  3.644960e+03
      drops  2.722000e+01  2.211814e+03
rankCountry  1.656600

### F.4 Histograms and boxplots

In [45]:
# Fetch histogram data for key columns via DuckDB histogram bins
# matches_raw.rating
rating_hist_sql = """
SELECT
    (FLOOR(rating / 100) * 100)::INTEGER AS bin,
    COUNT(*) AS cnt
FROM matches_raw
WHERE rating IS NOT NULL
GROUP BY bin
ORDER BY bin
"""
rating_hist_df = con.execute(rating_hist_sql).df()

In [46]:
# matches_raw.ratingDiff
ratingdiff_hist_sql = """
SELECT
    (FLOOR("ratingDiff" / 10) * 10)::INTEGER AS bin,
    COUNT(*) AS cnt
FROM matches_raw
WHERE "ratingDiff" IS NOT NULL
GROUP BY bin
ORDER BY bin
"""
ratingdiff_hist_df = con.execute(ratingdiff_hist_sql).df()

In [47]:
# ratings_raw.rating
ratings_rating_hist_sql = """
SELECT
    (FLOOR(rating / 100) * 100)::INTEGER AS bin,
    COUNT(*) AS cnt
FROM ratings_raw
WHERE rating IS NOT NULL
GROUP BY bin
ORDER BY bin
"""
ratings_rating_hist_df = con.execute(ratings_rating_hist_sql).df()

In [48]:
# leaderboards_raw.rating
lb_rating_hist_sql = """
SELECT
    (FLOOR(rating / 100) * 100)::INTEGER AS bin,
    COUNT(*) AS cnt
FROM leaderboards_raw
WHERE rating IS NOT NULL
GROUP BY bin
ORDER BY bin
"""
lb_rating_hist_df = con.execute(lb_rating_hist_sql).df()

In [49]:
# Verification: rating histograms (deferred to 01_02_05 for plotting)
print("=== matches_raw.rating histogram bins (feeds histogram in 01_02_05) ===")
print(rating_hist_df.to_string(index=False))
print("\n=== matches_raw.ratingDiff histogram bins (feeds histogram in 01_02_05) ===")
print(ratingdiff_hist_df.to_string(index=False))
print("\n=== ratings_raw.rating histogram bins (feeds histogram in 01_02_05) ===")
print(ratings_rating_hist_df.to_string(index=False))
print("\n=== leaderboards_raw.rating histogram bins (feeds histogram in 01_02_05) ===")
print(lb_rating_hist_df.to_string(index=False))

=== matches_raw.rating histogram bins (feeds histogram in 01_02_05) ===
 bin      cnt
-100      534
   0    46951
 100    78568
 200   171650
 300   396362
 400   980370
 500  2312309
 600  4769442
 700  8949311
 800 15461489
 900 21956782
1000 26136311
1100 20651170
1200 17804317
1300 14988617
1400 10663652
1500  6197143
1600  3184322
1700  1746394
1800  1030710
1900   631429
2000   394760
2100   300708
2200   212480
2300   143368
2400   109482
2500    64764
2600    35788
2700    20880
2800     9226
2900     3982
3000     1560
3100      641
3200      386
3300      305
3400      146
3500       90
3600       97
3700       35
3800       11
3900       20
4000        7
4100        9
4200        5
4300        2
4400        2
4500        2
4800        2
4900        1
5000        1

=== matches_raw.ratingDiff histogram bins (feeds histogram in 01_02_05) ===
 bin      cnt
-180        3
-170       63
-160      122
-150      210
-140      343
-130      488
-120      844
-110     1768
-100     40

In [50]:
# Verification: boxplot percentile data for rating and ratingDiff (deferred to 01_02_05)
boxplot_data = []
boxplot_labels = []
for stats_list, prefix in [
    (matches_numeric_stats, "m."),
    (ratings_numeric_stats, "r."),
    (lb_numeric_stats, "lb."),
]:
    for s in stats_list:
        if s["column_name"] in ("rating", "ratingDiff"):
            boxplot_labels.append(f"{prefix}{s['column_name']}")
            boxplot_data.append(s)

print("=== Numeric boxplot data (p05/p25/p50/p75/p95) -- feeds boxplot in 01_02_05 ===")
boxplot_summary_df = pd.DataFrame([
    {
        "label": label,
        "p05": s["p05"],
        "p25": s["p25"],
        "median": s["median_val"],
        "p75": s["p75"],
        "p95": s["p95"],
    }
    for label, s in zip(boxplot_labels, boxplot_data)
])
print(boxplot_summary_df.to_string(index=False))

=== Numeric boxplot data (p05/p25/p50/p75/p95) -- feeds boxplot in 01_02_05 ===
       label   p05   p25  median    p75    p95
    m.rating 687.0 934.0  1093.0 1299.0 1598.0
m.ratingDiff -20.0 -16.0     0.0   16.0   20.0
    r.rating 693.0 974.0  1093.0 1286.0 1552.0
   lb.rating 746.0 953.0  1003.0 1111.0 1408.0


## G. Temporal range

In [51]:
temporal_range_sql = """
SELECT
    MIN(started) AS earliest_match,
    MAX(started) AS latest_match,
    MIN(finished) AS earliest_finish,
    MAX(finished) AS latest_finish,
    COUNT(DISTINCT CAST(started AS DATE)) AS distinct_match_dates
FROM matches_raw
"""
print("Running temporal range query...")
temporal_df = con.execute(temporal_range_sql).df()
print("\n=== Temporal range (matches_raw) ===")
print(temporal_df.to_string(index=False))
findings["temporal_range_matches"] = temporal_df.to_dict(orient="records")

Running temporal range query...



=== Temporal range (matches_raw) ===
     earliest_match        latest_match     earliest_finish       latest_finish  distinct_match_dates
2020-07-31 13:32:14 2026-04-04 23:58:58 2020-08-01 00:00:11 2026-04-04 23:59:59                  2074


In [52]:
ratings_temporal_sql = """
SELECT
    MIN(date) AS earliest_rating,
    MAX(date) AS latest_rating,
    COUNT(DISTINCT CAST(date AS DATE)) AS distinct_rating_dates
FROM ratings_raw
"""
ratings_temporal_df = con.execute(ratings_temporal_sql).df()
print("\n=== Temporal range (ratings_raw) ===")
print(ratings_temporal_df.to_string(index=False))
findings["temporal_range_ratings"] = ratings_temporal_df.to_dict(orient="records")


=== Temporal range (ratings_raw) ===
earliest_rating       latest_rating  distinct_rating_dates
     2022-09-10 2026-04-03 23:59:59                    735


In [53]:
duration_sql = """
SELECT
    ROUND(AVG(EXTRACT(EPOCH FROM (finished - started))), 2) AS avg_duration_secs,
    ROUND(MEDIAN(EXTRACT(EPOCH FROM (finished - started))), 2)
        AS median_duration_secs,
    MIN(EXTRACT(EPOCH FROM (finished - started))) AS min_duration_secs,
    MAX(EXTRACT(EPOCH FROM (finished - started))) AS max_duration_secs,
    PERCENTILE_CONT(0.05) WITHIN GROUP
        (ORDER BY EXTRACT(EPOCH FROM (finished - started))) AS p05_secs,
    PERCENTILE_CONT(0.95) WITHIN GROUP
        (ORDER BY EXTRACT(EPOCH FROM (finished - started))) AS p95_secs
FROM matches_raw
WHERE finished > started
"""
print("Running match duration distribution query...")
duration_df = con.execute(duration_sql).df()
print("\n=== Match duration distribution (seconds) ===")
print(duration_df.to_string(index=False))
findings["match_duration_stats"] = duration_df.to_dict(orient="records")

Running match duration distribution query...



=== Match duration distribution (seconds) ===
 avg_duration_secs  median_duration_secs  min_duration_secs  max_duration_secs  p05_secs  p95_secs
           1814.87                1678.0                1.0          3279303.0     145.0    3789.0


In [54]:
# Excluded rows count
excluded_sql = """
SELECT
    COUNT(*) FILTER (WHERE finished IS NULL OR started IS NULL)
        AS null_timestamp_count,
    COUNT(*) FILTER (WHERE finished IS NOT NULL AND started IS NOT NULL
        AND finished <= started)
        AS non_positive_duration_count
FROM matches_raw
"""
excluded_df = con.execute(excluded_sql).df()
print("\n=== Excluded from duration calculation ===")
print(excluded_df.to_string(index=False))
findings["duration_excluded_rows"] = excluded_df.to_dict(orient="records")


=== Excluded from duration calculation ===
 null_timestamp_count  non_positive_duration_count
                    0                         2941


In [55]:
# Time-series: monthly match counts
monthly_sql = """
SELECT
    DATE_TRUNC('month', started) AS month,
    COUNT(DISTINCT matchId) AS distinct_matches,
    COUNT(*) AS total_rows
FROM matches_raw
WHERE started IS NOT NULL
GROUP BY month
ORDER BY month
"""
monthly_df = con.execute(monthly_sql).df()
print(f"\nMonthly buckets: {len(monthly_df)} months")


Monthly buckets: 70 months


In [56]:
# Verification: monthly match counts (deferred to 01_02_05 for line chart)
print("=== Monthly match counts (feeds line chart in 01_02_05) ===")
print(monthly_df.to_string(index=False))

=== Monthly match counts (feeds line chart in 01_02_05) ===
     month  distinct_matches  total_rows
2020-07-01               135         665
2020-08-01            149946      679650
2020-09-01            147402      664976
2020-10-01            146176      651647
2020-11-01            241340     1087742
2020-12-01            281311     1226972
2021-01-01            346526     1511456
2021-02-01            277219     1210048
2021-03-01            280682     1214182
2021-04-01            329082     1430232
2021-05-01            340682     1510797
2021-06-01            270327     1194341
2021-07-01            313536     1346163
2021-08-01            298500     1293589
2021-09-01            295895     1263765
2021-10-01            312430     1341854
2021-11-01            241157     1030938
2021-12-01            255638     1106700
2022-01-01            308644     1364342
2022-02-01            308967     1320660
2022-03-01            301658     1330010
2022-04-01            369451     16243

## H. Auxiliary table NULL census

### H.1 leaderboards_raw (19 columns)

In [57]:
print("Running SUMMARIZE on leaderboards_raw...")
lb_summarize = con.execute("SUMMARIZE leaderboards_raw").df()
lb_total = con.execute(
    "SELECT COUNT(*) FROM leaderboards_raw"
).fetchone()[0]
print(f"leaderboards_raw total rows: {lb_total:,}")

Running SUMMARIZE on leaderboards_raw...


leaderboards_raw total rows: 2,381,227


In [58]:
lb_null_census = lb_summarize[["column_name", "null_percentage"]].copy()
lb_null_census["total_rows"] = lb_total
lb_null_census["null_count"] = (
    (lb_null_census["null_percentage"] / 100.0 * lb_total).round(0).astype(int)
)
lb_null_census["null_pct"] = lb_null_census["null_percentage"]
lb_null_census["approx_cardinality"] = lb_summarize["approx_unique"].astype(int)

print("\n=== leaderboards_raw NULL census (all 19 columns) ===")
print(
    lb_null_census[
        ["column_name", "total_rows", "null_count", "null_pct", "approx_cardinality"]
    ].to_string(index=False)
)


=== leaderboards_raw NULL census (all 19 columns) ===
  column_name  total_rows  null_count  null_pct  approx_cardinality
  leaderboard     2381227           0      0.00                  17
    profileId     2381227           0      0.00             2168633
         name     2381227           0      0.00             1286596
         rank     2381227      609832     25.61              201232
       rating     2381227           0      0.00                2682
lastMatchTime     2381227           0      0.00             1594116
        drops     2381227      609832     25.61                1085
       losses     2381227      609832     25.61                4284
       streak     2381227      609832     25.61                 337
         wins     2381227      609832     25.61                5022
        games     2381227           0      0.00                7234
    updatedAt     2381227           0      0.00               47729
  rankCountry     2381227      666505     27.99              

In [59]:
findings["leaderboards_raw_null_census"] = (
    lb_null_census[
        ["column_name", "total_rows", "null_count", "null_pct", "approx_cardinality"]
    ].to_dict(orient="records")
)
findings["leaderboards_raw_total_rows"] = int(lb_total)

### H.2 profiles_raw (14 columns)

In [60]:
print("Running SUMMARIZE on profiles_raw...")
pr_summarize = con.execute("SUMMARIZE profiles_raw").df()
pr_total = con.execute(
    "SELECT COUNT(*) FROM profiles_raw"
).fetchone()[0]
print(f"profiles_raw total rows: {pr_total:,}")

Running SUMMARIZE on profiles_raw...


profiles_raw total rows: 3,609,686


In [61]:
pr_null_census = pr_summarize[["column_name", "null_percentage"]].copy()
pr_null_census["total_rows"] = pr_total
pr_null_census["null_count"] = (
    (pr_null_census["null_percentage"] / 100.0 * pr_total).round(0).astype(int)
)
pr_null_census["null_pct"] = pr_null_census["null_percentage"]
pr_null_census["approx_cardinality"] = pr_summarize["approx_unique"].astype(int)

print("\n=== profiles_raw NULL census (all 14 columns) ===")
print(
    pr_null_census[
        ["column_name", "total_rows", "null_count", "null_pct", "approx_cardinality"]
    ].to_string(index=False)
)


=== profiles_raw NULL census (all 14 columns) ===
       column_name  total_rows  null_count  null_pct  approx_cardinality
         profileId     3609686           0      0.00             4093912
           steamId     3609686        1805      0.05             3668366
              name     3609686           0      0.00             3023244
              clan     3609686           0      0.00              136315
           country     3609686     1604144     44.44                 237
        avatarhash     3609686       32487      0.90             1658324
     sharedHistory     3609686     3609686    100.00                   2
     twitchChannel     3609686     3609686    100.00                  43
    youtubeChannel     3609686     3609686    100.00                   9
youtubeChannelName     3609686     3609686    100.00                   9
         discordId     3609686     3609686    100.00                  44
       discordName     3609686     3609686    100.00                  44


In [62]:
findings["profiles_raw_null_census"] = (
    pr_null_census[
        ["column_name", "total_rows", "null_count", "null_pct", "approx_cardinality"]
    ].to_dict(orient="records")
)
findings["profiles_raw_total_rows"] = int(pr_total)

### H.3 ratings_raw (8 columns)

In [63]:
print("Running SUMMARIZE on ratings_raw...")
rt_summarize = con.execute("SUMMARIZE ratings_raw").df()
rt_total = con.execute(
    "SELECT COUNT(*) FROM ratings_raw"
).fetchone()[0]
print(f"ratings_raw total rows: {rt_total:,}")

Running SUMMARIZE on ratings_raw...


ratings_raw total rows: 58,317,433


In [64]:
rt_null_census = rt_summarize[["column_name", "null_percentage"]].copy()
rt_null_census["total_rows"] = rt_total
rt_null_census["null_count"] = (
    (rt_null_census["null_percentage"] / 100.0 * rt_total).round(0).astype(int)
)
rt_null_census["null_pct"] = rt_null_census["null_percentage"]
rt_null_census["approx_cardinality"] = rt_summarize["approx_unique"].astype(int)

print("\n=== ratings_raw NULL census (all 8 columns) ===")
print(
    rt_null_census[
        ["column_name", "total_rows", "null_count", "null_pct", "approx_cardinality"]
    ].to_string(index=False)
)


=== ratings_raw NULL census (all 8 columns) ===
   column_name  total_rows  null_count  null_pct  approx_cardinality
    profile_id    58317433           0      0.00              819485
         games    58317433           0      0.00               42315
        rating    58317433           0      0.00                3100
          date    58317433           0      0.00            12435988
leaderboard_id    58317433           0      0.00                  15
   rating_diff    58317433     1078873      1.85                 222
        season    58317433           0      0.00                   1
      filename    58317433           0      0.00                 545


In [65]:
findings["ratings_raw_null_census"] = (
    rt_null_census[
        ["column_name", "total_rows", "null_count", "null_pct", "approx_cardinality"]
    ].to_dict(orient="records")
)
findings["ratings_raw_total_rows"] = int(rt_total)

### H.1b leaderboards_raw categorical, boolean, and temporal

T03: leaderboards_raw categorical (leaderboard, country), boolean (active),
and temporal (lastMatchTime, updatedAt) profiling.

In [66]:
# T03: leaderboard VARCHAR -- all values (low cardinality)
lb_leaderboard_sql = """
SELECT
    leaderboard AS value,
    COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM leaderboards_raw
GROUP BY leaderboard
ORDER BY cnt DESC
"""
lb_leaderboard_df = con.execute(lb_leaderboard_sql).df()
print("=== leaderboards_raw.leaderboard distribution ===")
print(lb_leaderboard_df.to_string(index=False))

=== leaderboards_raw.leaderboard distribution ===
                       value     cnt   pct
                    unranked 1729627 72.64
                     rm_team  329438 13.83
                      rm_1v1  242054 10.17
                     ew_team   29593  1.24
                      ew_1v1   17061  0.72
             rm_team_console   12938  0.54
              rm_1v1_console    8935  0.38
                     unknown    7118  0.30
        ew_1v1_redbullwololo    3599  0.15
                     ror_1v1     357  0.01
                    ror_team     340  0.01
                     dm_team      81  0.00
                      dm_1v1      38  0.00
              ew_1v1_console      25  0.00
             ew_team_console      21  0.00
ew_1v1_redbullwololo_console       2  0.00


In [67]:
# T03: country VARCHAR -- top 30 + NULL count
lb_country_sql = """
SELECT country AS value, COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM leaderboards_raw GROUP BY country ORDER BY cnt DESC LIMIT 30
"""
lb_country_df = con.execute(lb_country_sql).df()
lb_country_null_count = con.execute(
    "SELECT COUNT(*) - COUNT(country) FROM leaderboards_raw"
).fetchone()[0]
print(f"\n=== leaderboards_raw.country top 30 (null_count={lb_country_null_count:,}) ===")
print(lb_country_df.to_string(index=False))


=== leaderboards_raw.country top 30 (null_count=760,503) ===
value    cnt   pct
  NaN 760503 31.94
   us 217269  9.12
   de 149736  6.29
   cn 147507  6.19
   ar 114274  4.80
   fr  74391  3.12
   tw  62411  2.62
   tr  61952  2.60
   mx  60248  2.53
   cl  59733  2.51
   gb  55359  2.32
   es  51590  2.17
   br  50458  2.12
   au  47333  1.99
   ca  44117  1.85
   co  30992  1.30
   nl  26504  1.11
   cz  25931  1.09
   ru  24932  1.05
   in  19746  0.83
   se  18492  0.78
   it  18387  0.77
   pe  18012  0.76
   hk  17621  0.74
   ch  16387  0.69
   pl  15116  0.63
   be  14261  0.60
   at  12469  0.52
   nz  10122  0.43
   ro   9580  0.40


In [68]:
# T03: boolean census for active
lb_boolean_sql = """
SELECT 'active' AS column_name,
    COUNT(*) FILTER (WHERE active = TRUE) AS true_count,
    COUNT(*) FILTER (WHERE active = FALSE) AS false_count,
    COUNT(*) - COUNT(active) AS null_count,
    COUNT(*) AS total_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE active = TRUE) / COUNT(*), 2) AS true_pct,
    ROUND(100.0 * COUNT(*) FILTER (WHERE active = FALSE) / COUNT(*), 2) AS false_pct,
    ROUND(100.0 * (COUNT(*) - COUNT(active)) / COUNT(*), 2) AS null_pct
FROM leaderboards_raw
"""
lb_boolean_df = con.execute(lb_boolean_sql).df()
print("\n=== leaderboards_raw boolean census (active) ===")
print(lb_boolean_df.to_string(index=False))


=== leaderboards_raw boolean census (active) ===
column_name  true_count  false_count  null_count  total_rows  true_pct  false_pct  null_pct
     active      317789      1453672      609766     2381227     13.35      61.05     25.61


In [69]:
# T03: temporal range for lastMatchTime and updatedAt
lb_temporal_sql = """
SELECT
    'lastMatchTime' AS column_name,
    MIN("lastMatchTime") AS min_val,
    MAX("lastMatchTime") AS max_val
FROM leaderboards_raw
UNION ALL
SELECT
    'updatedAt' AS column_name,
    MIN("updatedAt") AS min_val,
    MAX("updatedAt") AS max_val
FROM leaderboards_raw
"""
lb_temporal_df = con.execute(lb_temporal_sql).df()
print("\n=== leaderboards_raw temporal ranges ===")
print(lb_temporal_df.to_string(index=False))


=== leaderboards_raw temporal ranges ===
  column_name                 min_val                 max_val
lastMatchTime 2022-09-06 18:47:29.000 2026-04-06 00:12:25.000
    updatedAt 2023-08-23 22:05:17.362 2026-04-06 00:12:39.518


In [70]:
# T03: ratings_raw leaderboard_id distribution
rt_lb_id_sql = """
SELECT leaderboard_id AS value, COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM ratings_raw GROUP BY leaderboard_id ORDER BY cnt DESC
"""
rt_lb_id_df = con.execute(rt_lb_id_sql).df()
print("\n=== ratings_raw.leaderboard_id distribution ===")
print(rt_lb_id_df.to_string(index=False))


=== ratings_raw.leaderboard_id distribution ===
 value      cnt   pct
     0 25839038 44.31
     4 18853083 32.33
     3 11784760 20.21
    14   563025  0.97
    13   387207  0.66
    16   364138  0.62
    15   342918  0.59
    29   104140  0.18
  5019    53557  0.09
  5018    25366  0.04
    18       89  0.00
    25       61  0.00
    17       44  0.00
    30        5  0.00
    27        2  0.00


In [71]:
findings["leaderboards_raw_categorical"] = {
    "leaderboard": lb_leaderboard_df.to_dict(orient="records"),
    "country": {
        "top_30": lb_country_df.to_dict(orient="records"),
        "null_count": int(lb_country_null_count),
    },
}
findings["leaderboards_raw_boolean"] = lb_boolean_df.to_dict(orient="records")
findings["leaderboards_raw_temporal"] = lb_temporal_df.to_dict(orient="records")
findings["ratings_raw_leaderboard_id_distribution"] = rt_lb_id_df.to_dict(orient="records")

### H.2b profiles_raw categorical

T04: categorical profiling for profiles_raw country and clan columns.
Note: 7 dead columns (sharedHistory, twitchChannel, youtubeChannel,
youtubeChannelName, discordId, discordName, discordInvitation) are documented
in constant_fields (see Section I).

In [72]:
# T04: country VARCHAR -- top 30 + NULL count and NULL pct
pr_country_sql = """
SELECT country AS value, COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM profiles_raw GROUP BY country ORDER BY cnt DESC LIMIT 30
"""
pr_country_df = con.execute(pr_country_sql).df()
pr_country_null_count = con.execute(
    "SELECT COUNT(*) - COUNT(country) FROM profiles_raw"
).fetchone()[0]
pr_country_null_pct = round(100.0 * pr_country_null_count / pr_total, 2)
print(f"=== profiles_raw.country top 30 (null_count={pr_country_null_count:,}, null_pct={pr_country_null_pct}%) ===")
print(pr_country_df.to_string(index=False))

=== profiles_raw.country top 30 (null_count=1,604,091, null_pct=44.44%) ===
value     cnt   pct
  NaN 1604091 44.44
   us  270310  7.49
   de  197117  5.46
   cn  194662  5.39
   ar  129261  3.58
   fr   98939  2.74
   tw   76099  2.11
   tr   75176  2.08
   cl   72553  2.01
   br   71678  1.99
   mx   64933  1.80
   gb   64714  1.79
   au   64469  1.79
   ca   56797  1.57
   es   56083  1.55
   ru   35766  0.99
   nl   32858  0.91
   cz   31744  0.88
   co   29551  0.82
   se   23672  0.66
   it   21813  0.60
   hk   21715  0.60
   in   21101  0.58
   pl   20589  0.57
   ch   19291  0.53
   be   18150  0.50
   pe   18050  0.50
   at   16020  0.44
   nz   12392  0.34
   ro   11975  0.33


In [73]:
# T04: clan VARCHAR -- top 30 + cardinality + null count
pr_clan_sql = """
SELECT clan AS value, COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM profiles_raw GROUP BY clan ORDER BY cnt DESC LIMIT 30
"""
pr_clan_df = con.execute(pr_clan_sql).df()
pr_clan_cardinality = con.execute(
    "SELECT COUNT(DISTINCT clan) FROM profiles_raw"
).fetchone()[0]
pr_clan_null_count = con.execute(
    "SELECT COUNT(*) - COUNT(clan) FROM profiles_raw"
).fetchone()[0]
print(f"\n=== profiles_raw.clan top 30 (cardinality={pr_clan_cardinality:,}, null_count={pr_clan_null_count:,}) ===")
print(pr_clan_df.to_string(index=False))


=== profiles_raw.clan top 30 (cardinality=94,173, null_count=3) ===
value     cnt   pct
      3249708 90.03
CAT00     197  0.01
 CoZa     197  0.01
Yeome     196  0.01
 ClTg     196  0.01
Katik     195  0.01
STURM     193  0.01
WeBFF     193  0.01
CreSF     191  0.01
 TWWS     191  0.01
 L501     190  0.01
VZLAA     190  0.01
 KDLD     190  0.01
BLKLN     189  0.01
BRAZ1     189  0.01
 NaCl     188  0.01
OCPOT     188  0.01
 XXNB     188  0.01
TRAOE     187  0.01
ROTLD     186  0.01
 RePo     186  0.01
 Albi     185  0.01
 DOGE     185  0.01
  BaR     184  0.01
  BUL     184  0.01
0ARG0     183  0.01
 VBPK     183  0.01
  JZD     182  0.01
  VNS     182  0.01
TWCEA     182  0.01


In [74]:
findings["profiles_raw_categorical"] = {
    "country": {
        "top_30": pr_country_df.to_dict(orient="records"),
        "null_count": int(pr_country_null_count),
        "null_pct": pr_country_null_pct,
    },
    "clan": {
        "top_30": pr_clan_df.to_dict(orient="records"),
        "cardinality": int(pr_clan_cardinality),
        "null_count": int(pr_clan_null_count),
    },
}

## I2. Duplicate row detection

T05: Check for duplicate composite keys in matches_raw and ratings_raw.

In [75]:
# T05: matches_raw -- duplicate (matchId, profileId) pairs
dup_matches_sql = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CAST("matchId" AS VARCHAR) || '|' || CAST("profileId" AS VARCHAR))
        AS distinct_pairs,
    COUNT(*) - COUNT(DISTINCT CAST("matchId" AS VARCHAR) || '|' || CAST("profileId" AS VARCHAR))
        AS duplicate_rows
FROM matches_raw
"""
dup_matches_df = con.execute(dup_matches_sql).df()
print("=== matches_raw duplicate check (matchId, profileId) ===")
print(dup_matches_df.to_string(index=False))

=== matches_raw duplicate check (matchId, profileId) ===
 total_rows  distinct_pairs  duplicate_rows
  277099059       268287054         8812005


In [76]:
# T05: ratings_raw -- duplicate (profile_id, leaderboard_id, date) triples
dup_ratings_sql = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CAST(profile_id AS VARCHAR) || '|'
        || CAST(leaderboard_id AS VARCHAR) || '|'
        || CAST(date AS VARCHAR))
        AS distinct_triples,
    COUNT(*) - COUNT(DISTINCT CAST(profile_id AS VARCHAR) || '|'
        || CAST(leaderboard_id AS VARCHAR) || '|'
        || CAST(date AS VARCHAR))
        AS duplicate_rows
FROM ratings_raw
"""
dup_ratings_df = con.execute(dup_ratings_sql).df()
print("\n=== ratings_raw duplicate check (profile_id, leaderboard_id, date) ===")
print(dup_ratings_df.to_string(index=False))


=== ratings_raw duplicate check (profile_id, leaderboard_id, date) ===
 total_rows  distinct_triples  duplicate_rows
   58317433          58317433               0


In [77]:
findings["matches_raw_duplicate_check"] = dup_matches_df.to_dict(orient="records")
findings["ratings_raw_duplicate_check"] = dup_ratings_df.to_dict(orient="records")

## I3. NULL co-occurrence for 0.15%-0.16% clusters

T06: Two distinct NULL clusters in matches_raw:
- Cluster A (8 cols, null_count=415,649): allowCheats, lockSpeed, lockTeams,
  recordGame, sharedExploration, teamPositions, teamTogether, turboMode
- Cluster B (2 cols, null_count=443,358): fullTechTree, population
Note: fullTechTree and population have a different null count from the 8-column
cluster -- these are two separate clusters.

In [78]:
# T06: Cluster A -- are all 8 columns simultaneously NULL?
null_cooc_a_sql = """
SELECT
    COUNT(*) AS all_eight_null_simultaneously,
    (SELECT COUNT(*) FROM matches_raw WHERE "allowCheats" IS NULL)
        AS allowCheats_null_count
FROM matches_raw
WHERE "allowCheats" IS NULL
    AND "lockSpeed" IS NULL AND "lockTeams" IS NULL
    AND "recordGame" IS NULL AND "sharedExploration" IS NULL
    AND "teamPositions" IS NULL AND "teamTogether" IS NULL
    AND "turboMode" IS NULL
"""
null_cooc_a_df = con.execute(null_cooc_a_sql).df()
print("=== Cluster A NULL co-occurrence (8 columns) ===")
print(null_cooc_a_df.to_string(index=False))

=== Cluster A NULL co-occurrence (8 columns) ===
 all_eight_null_simultaneously  allowCheats_null_count
                        426472                  428338


In [79]:
# T06: Cluster B -- are fullTechTree and population simultaneously NULL?
null_cooc_b_sql = """
SELECT
    COUNT(*) AS both_null,
    (SELECT COUNT(*) FROM matches_raw WHERE "fullTechTree" IS NULL)
        AS fullTechTree_null_count,
    (SELECT COUNT(*) FROM matches_raw WHERE population IS NULL)
        AS population_null_count
FROM matches_raw
WHERE "fullTechTree" IS NULL AND population IS NULL
"""
null_cooc_b_df = con.execute(null_cooc_b_sql).df()
print("\n=== Cluster B NULL co-occurrence (fullTechTree, population) ===")
print(null_cooc_b_df.to_string(index=False))


=== Cluster B NULL co-occurrence (fullTechTree, population) ===
 both_null  fullTechTree_null_count  population_null_count
    431288                   431494                 431290


In [80]:
# T06: Cross-cluster overlap -- are Cluster A NULLs a subset of Cluster B NULLs?
null_cooc_cross_sql = """
SELECT
    COUNT(*) FILTER (WHERE "allowCheats" IS NULL AND "fullTechTree" IS NULL)
        AS both_clusters_null,
    COUNT(*) FILTER (WHERE "allowCheats" IS NULL AND "fullTechTree" IS NOT NULL)
        AS cluster_a_only_null,
    COUNT(*) FILTER (WHERE "allowCheats" IS NOT NULL AND "fullTechTree" IS NULL)
        AS cluster_b_only_null
FROM matches_raw
"""
null_cooc_cross_df = con.execute(null_cooc_cross_sql).df()
print("\n=== Cross-cluster overlap (Cluster A vs Cluster B) ===")
print(null_cooc_cross_df.to_string(index=False))


=== Cross-cluster overlap (Cluster A vs Cluster B) ===
 both_clusters_null  cluster_a_only_null  cluster_b_only_null
             428321                   17                 3173


In [81]:
findings["matches_raw_null_cooccurrence"] = {
    "cluster_a_eight_cols": null_cooc_a_df.to_dict(orient="records"),
    "cluster_b_fulltree_population": null_cooc_b_df.to_dict(orient="records"),
    "cross_cluster_overlap": null_cooc_cross_df.to_dict(orient="records"),
}

## Post-game field annotations (Invariant #3)

Columns that encode match outcome -- using these as prediction features would
be temporal leakage.

In [82]:
findings["post_game_fields"] = [
    {
        "table": "matches_raw",
        "column": "ratingDiff",
        "type": "INTEGER",
        "reason": (
            "Rating change resulting from match outcome. Using this to predict "
            "the match it encodes would be temporal leakage (Invariant #3)."
        )
    },
    {
        "table": "ratings_raw",
        "column": "rating_diff",
        "type": "BIGINT",
        "reason": (
            "Rating change resulting from match outcome. Semantically identical "
            "to matches_raw.ratingDiff. Using this to predict the match it "
            "encodes would be temporal leakage (Invariant #3)."
        )
    },
    {
        "table": "matches_raw",
        "column": "won",
        "type": "BOOLEAN",
        "reason": "Prediction target. Post-game by definition."
    },
    {
        "table": "matches_raw",
        "column": "finished",
        "type": "TIMESTAMP",
        "reason": "Match end timestamp. Known only after match completes."
    },
]
print("Post-game field annotations recorded.")

Post-game field annotations recorded.


## Field Classification (preliminary)

Preliminary pre-game/post-game classification for all 55 matches_raw columns.
Formal boundary deferred to Phase 02 (Feature Engineering).

In [83]:
field_classification = {
    "table": "matches_raw",
    "classification_status": "preliminary",
    "formal_boundary_deferred_to": "Phase 02 (Feature Engineering)",
    "classification_notes": {
        "pre_game": "Known before match starts. Usable as prediction features without temporal leakage.",
        "post_game": "Known only after match ends. Using as feature is temporal leakage (Invariant #3).",
        "ambiguous_pre_or_post": "Temporal status unknown; requires Phase 02 investigation before use.",
        "identifier": "Player/match identity columns. Not features.",
        "target": "Prediction target. Never a feature.",
        "context": "Player-level metadata with unclear temporal availability; not match settings and not match outcomes.",
    },
    "fields": [
        # Identifiers
        {"column": "matchId",              "temporal_class": "identifier",            "notes": "Match identifier"},
        {"column": "profileId",            "temporal_class": "identifier",            "notes": "Player identifier"},
        {"column": "name",                 "temporal_class": "identifier",            "notes": "Player name"},
        {"column": "filename",             "temporal_class": "identifier",            "notes": "Source file provenance (I10)"},
        # Target
        {"column": "won",                  "temporal_class": "target",               "notes": "Prediction target"},
        # Post-game
        {"column": "ratingDiff",           "temporal_class": "post_game",            "notes": "Rating change from match outcome (Invariant #3)"},
        {"column": "finished",             "temporal_class": "post_game",            "notes": "Match end timestamp"},
        # Ambiguous -- needs Phase 02 investigation
        {"column": "rating",               "temporal_class": "ambiguous_pre_or_post", "notes": "Could be pre-match or post-match snapshot -- identical 42.46% NULL rate suggests co-occurrence with ratingDiff (unverified; needs row-level check)"},
        # Pre-game (match settings)
        {"column": "started",              "temporal_class": "pre_game",             "notes": "Match start timestamp"},
        {"column": "leaderboard",          "temporal_class": "pre_game",             "notes": "Ranked queue / game mode"},
        {"column": "internalLeaderboardId","temporal_class": "pre_game",             "notes": "Numeric leaderboard ID"},
        {"column": "map",                  "temporal_class": "pre_game",             "notes": "Map selection"},
        {"column": "mapSize",              "temporal_class": "pre_game",             "notes": "Map size setting"},
        {"column": "civ",                  "temporal_class": "pre_game",             "notes": "Civilization selection"},
        {"column": "gameMode",             "temporal_class": "pre_game",             "notes": "Game mode"},
        {"column": "gameVariant",          "temporal_class": "pre_game",             "notes": "Game variant"},
        {"column": "speed",                "temporal_class": "pre_game",             "notes": "Game speed setting"},
        {"column": "speedFactor",          "temporal_class": "pre_game",             "notes": "Speed multiplier"},
        {"column": "population",           "temporal_class": "pre_game",             "notes": "Population cap"},
        {"column": "resources",            "temporal_class": "pre_game",             "notes": "Resource setting"},
        {"column": "startingAge",          "temporal_class": "pre_game",             "notes": "Starting age setting"},
        {"column": "endingAge",            "temporal_class": "pre_game",             "notes": "Ending age setting"},
        {"column": "victory",              "temporal_class": "pre_game",             "notes": "Victory condition"},
        {"column": "difficulty",           "temporal_class": "pre_game",             "notes": "AI difficulty setting"},
        {"column": "civilizationSet",      "temporal_class": "pre_game",             "notes": "Civ set restriction"},
        {"column": "revealMap",            "temporal_class": "pre_game",             "notes": "Map visibility setting"},
        {"column": "treatyLength",         "temporal_class": "pre_game",             "notes": "Treaty length setting"},
        {"column": "mod",                  "temporal_class": "pre_game",             "notes": "Mod enabled flag"},
        {"column": "fullTechTree",         "temporal_class": "pre_game",             "notes": "Full tech tree toggle"},
        {"column": "allowCheats",          "temporal_class": "pre_game",             "notes": "Cheats toggle"},
        {"column": "empireWarsMode",       "temporal_class": "pre_game",             "notes": "Empire Wars toggle"},
        {"column": "lockSpeed",            "temporal_class": "pre_game",             "notes": "Lock speed toggle"},
        {"column": "lockTeams",            "temporal_class": "pre_game",             "notes": "Lock teams toggle"},
        {"column": "hideCivs",             "temporal_class": "pre_game",             "notes": "Hide civs toggle"},
        {"column": "recordGame",           "temporal_class": "pre_game",             "notes": "Record game toggle"},
        {"column": "regicideMode",         "temporal_class": "pre_game",             "notes": "Regicide toggle"},
        {"column": "sharedExploration",    "temporal_class": "pre_game",             "notes": "Shared exploration toggle"},
        {"column": "suddenDeathMode",      "temporal_class": "pre_game",             "notes": "Sudden death toggle"},
        {"column": "antiquityMode",        "temporal_class": "pre_game",             "notes": "Antiquity mode toggle"},
        {"column": "teamPositions",        "temporal_class": "pre_game",             "notes": "Team positions toggle"},
        {"column": "teamTogether",         "temporal_class": "pre_game",             "notes": "Team together toggle"},
        {"column": "turboMode",            "temporal_class": "pre_game",             "notes": "Turbo mode toggle"},
        {"column": "color",                "temporal_class": "pre_game",             "notes": "Player color slot"},
        {"column": "colorHex",             "temporal_class": "pre_game",             "notes": "Player color hex code"},
        {"column": "slot",                 "temporal_class": "pre_game",             "notes": "Player slot number"},
        {"column": "team",                 "temporal_class": "pre_game",             "notes": "Team assignment"},
        {"column": "password",             "temporal_class": "pre_game",             "notes": "Password-protected match (BOOLEAN, 82.9% NULL)"},
        {"column": "scenario",             "temporal_class": "pre_game",             "notes": "Scenario name (98.27% NULL)"},
        {"column": "modDataset",           "temporal_class": "pre_game",             "notes": "Mod dataset name (99.72% NULL)"},
        # Context
        {"column": "server",               "temporal_class": "context",              "notes": "Server (97.99% NULL)"},
        {"column": "privacy",              "temporal_class": "context",              "notes": "Player privacy setting"},
        {"column": "status",               "temporal_class": "context",              "notes": "Player status"},
        {"column": "country",              "temporal_class": "context",              "notes": "Player country"},
        {"column": "shared",               "temporal_class": "context",              "notes": "Shared flag"},
        {"column": "verified",             "temporal_class": "context",              "notes": "Verified flag"},
    ]
}
findings["field_classification"] = field_classification
print(f"Field classification: {len(field_classification['fields'])} columns annotated")
assert len(field_classification["fields"]) == 55, (
    f"Expected 55 fields, got {len(field_classification['fields'])}"
)

Field classification: 55 columns annotated


## I. Dead/constant/near-constant field detection

- Cardinality = 1: constant (dead field)
- Uniqueness ratio < 0.001: near-constant
- Uniqueness ratio = COUNT(DISTINCT col) / COUNT(*) where denominator
  includes NULLs (note: NULL deflation in denominator)

In [84]:
# T01: Combine census data from all four tables, including null_pct for dead-field fix
all_census = []
for table_name, census_df, total in [
    ("matches_raw", null_census_matches, total_rows),
    ("leaderboards_raw", lb_null_census, lb_total),
    ("profiles_raw", pr_null_census, pr_total),
    ("ratings_raw", rt_null_census, rt_total),
]:
    subset = census_df[["column_name", "approx_cardinality", "null_pct"]].copy()
    subset["table"] = table_name
    subset["total_rows"] = total
    subset["uniqueness_ratio"] = subset["approx_cardinality"] / total
    all_census.append(subset)
all_census_df = pd.concat(all_census, ignore_index=True)

In [85]:
# T01: Dead-field detection uses (approx_cardinality <= 1) OR (null_pct >= 100.0)
# The OR condition catches profiles_raw columns that are 100% NULL but have phantom
# HyperLogLog cardinalities > 1 due to approximation error (Flajolet et al. 2007).
constant_fields = all_census_df[
    (all_census_df["approx_cardinality"] <= 1) | (all_census_df["null_pct"] >= 100.0)
]

print(f"Dead fields detected: {len(constant_fields)}")
for _, row in constant_fields.iterrows():
    print(
        f"  {row['table']}.{row['column_name']} "
        f"(approx_cardinality={row['approx_cardinality']}, null_pct={row['null_pct']})"
    )

Dead fields detected: 12
  leaderboards_raw.season (approx_cardinality=1, null_pct=0.0)
  leaderboards_raw.rankLevel (approx_cardinality=1, null_pct=25.61)
  leaderboards_raw.filename (approx_cardinality=1, null_pct=0.0)
  profiles_raw.sharedHistory (approx_cardinality=2, null_pct=100.0)
  profiles_raw.twitchChannel (approx_cardinality=43, null_pct=100.0)
  profiles_raw.youtubeChannel (approx_cardinality=9, null_pct=100.0)
  profiles_raw.youtubeChannelName (approx_cardinality=9, null_pct=100.0)
  profiles_raw.discordId (approx_cardinality=44, null_pct=100.0)
  profiles_raw.discordName (approx_cardinality=44, null_pct=100.0)
  profiles_raw.discordInvitation (approx_cardinality=6, null_pct=100.0)
  profiles_raw.filename (approx_cardinality=1, null_pct=0.0)
  ratings_raw.season (approx_cardinality=1, null_pct=0.0)


In [86]:
# T01: Assert the 7 profiles_raw dead columns are detected
expected_dead_profiles = {
    "sharedHistory", "twitchChannel", "youtubeChannel",
    "youtubeChannelName", "discordId", "discordName", "discordInvitation"
}
actual_dead_profiles = set(
    constant_fields.loc[constant_fields["table"] == "profiles_raw", "column_name"]
)
missing = expected_dead_profiles - actual_dead_profiles
assert not missing, f"BLOCKER: Dead profiles_raw columns missing: {missing}"
print(f"BLOCKER assertion passed: all 7 profiles_raw dead columns detected.")

BLOCKER assertion passed: all 7 profiles_raw dead columns detected.


In [87]:
# T08: Near-constant detection split into two buckets
# [I7] Empirical threshold: max clearly categorical cardinality is ~68 (civ);
# next meaningful categorical is map (~261) which falls above 100.
# Threshold of 100 cleanly separates low-cardinality semantically meaningful
# categoricals (leaderboard=22, civ=68) from map (261) and continuous columns.
# At N=277M rows, uniqueness_ratio < 0.001 flags all columns with < 277,000
# distinct values -- including semantically important categoricals.
NEAR_CONSTANT_CARDINALITY_THRESHOLD = 100  # empirical: max categorical=68 (civ), min ratio-flagged=261 (map)

# Bucket 1: genuinely low-cardinality (< 100 distinct values, not constant/dead)
near_constant_low_card = all_census_df[
    (all_census_df["uniqueness_ratio"] < 0.001)
    & (all_census_df["approx_cardinality"] > 1)
    & (all_census_df["approx_cardinality"] < NEAR_CONSTANT_CARDINALITY_THRESHOLD)
    & (all_census_df["null_pct"] < 100.0)
]
# Bucket 2: moderate-cardinality columns flagged only by ratio (NOT near-constant)
near_constant_ratio_flagged = all_census_df[
    (all_census_df["uniqueness_ratio"] < 0.001)
    & (all_census_df["approx_cardinality"] >= NEAR_CONSTANT_CARDINALITY_THRESHOLD)
]

print(f"\n=== Near-constant Bucket 1: low-cardinality (cardinality < {NEAR_CONSTANT_CARDINALITY_THRESHOLD}) ===")
if len(near_constant_low_card) > 0:
    print(near_constant_low_card.to_string(index=False))
else:
    print("None found.")

print(f"\n=== Near-constant Bucket 2: ratio-flagged only (cardinality >= {NEAR_CONSTANT_CARDINALITY_THRESHOLD}) ===")
print("(These include semantically important categoricals like map=261; NOT true near-constants.)")
if len(near_constant_ratio_flagged) > 0:
    print(near_constant_ratio_flagged.to_string(index=False))
else:
    print("None found.")


=== Near-constant Bucket 1: low-cardinality (cardinality < 100) ===
      column_name  approx_cardinality  null_pct            table  total_rows  uniqueness_ratio
      leaderboard                  22      0.00      matches_raw   277099059      7.939399e-08
           server                  11     97.99      matches_raw   277099059      3.969700e-08
          privacy                   3      0.00      matches_raw   277099059      1.082645e-08
              mod                   2      0.00      matches_raw   277099059      7.217635e-09
       difficulty                   7      0.00      matches_raw   277099059      2.526172e-08
      startingAge                  12      0.00      matches_raw   277099059      4.330581e-08
     fullTechTree                   2      0.16      matches_raw   277099059      7.217635e-09
      allowCheats                   2      0.15      matches_raw   277099059      7.217635e-09
   empireWarsMode                   2      1.06      matches_raw   277099059

In [88]:
findings["constant_fields"] = constant_fields.to_dict(orient="records")
findings["near_constant_low_cardinality"] = near_constant_low_card.to_dict(orient="records")
findings["near_constant_ratio_flagged"] = near_constant_ratio_flagged.to_dict(orient="records")

## J. Write artifacts

Write JSON and markdown artifacts with all findings and SQL.

In [89]:
# Serialization helper -- convert non-serializable types
json_str = json.dumps(findings, indent=2, default=str)
json_path = artifacts_dir / "01_02_04_univariate_census.json"
Path(json_path).write_text(json_str)
print(f"JSON artifact saved: {json_path}")
print(f"JSON size: {len(json_str):,} bytes")

JSON artifact saved: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/02_eda/01_02_04_univariate_census.json
JSON size: 105,622 bytes


In [90]:
md_lines = [
    "# Step 01_02_04 -- Univariate Census & Target Variable EDA",
    "",
    "**Dataset:** aoe2companion",
    "**Date:** 2026-04-14",
    "",
    "All SQL queries that produced reported results are inlined below (Invariant #6).",
    "",
    "---",
    "",
    "## A. Full NULL census of matches_raw",
    "",
    "```sql",
    "SUMMARIZE matches_raw",
    "SELECT COUNT(*) AS n FROM matches_raw",
    "```",
    "",
    "## A2. Empty-string investigation for VARCHAR columns",
    "",
    "Investigates whether `difficulty`, `colorHex`, `startingAge`, `endingAge`, `civilizationSet`",
    "(0 NULLs per SUMMARIZE) contain empty strings. Also confirms genuine NULLs for `scenario`",
    "and `modDataset`. `password` (BOOLEAN) is excluded -- empty-string hypothesis does not apply.",
    "",
    "```sql",
    "-- Per column (col in empty_string_cols):",
    "SELECT",
    "    '{col}' AS column_name,",
    "    COUNT(*) FILTER (WHERE \"{col}\" IS NULL)                                AS null_count,",
    "    COUNT(*) FILTER (WHERE \"{col}\" = '')                                   AS empty_string_count,",
    "    COUNT(*) FILTER (WHERE \"{col}\" IS NOT NULL AND \"{col}\" != '')          AS non_empty_count,",
    "    COUNT(*)                                                               AS total_rows",
    "FROM matches_raw",
    "```",
    "",
    "## B. Target variable (won)",
    "",
    "### won NULL count: exact GROUP BY vs SUMMARIZE approximation",
    "",
    "SUMMARIZE uses HyperLogLog approximation. The exact NULL count is taken from",
    "the GROUP BY distribution query and patched into `matches_raw_null_census`.",
    "",
    "### Overall distribution",
    "```sql",
    won_dist_sql.strip(),
    "```",
    "",
    "### Stratified by leaderboard",
    "```sql",
    won_by_lb_sql.strip(),
    "```",
    "",
    "### Intra-match consistency check (2-row matches)",
    "```sql",
    consistency_sql.strip(),
    "```",
    "",
    "## C. Match structure by leaderboard",
    "",
    "```sql",
    match_struct_sql.strip(),
    "```",
    "",
    "## D. Categorical field profiles",
    "",
    "```sql",
    '-- Per column (col in categorical list):',
    'SELECT',
    '    "{col}" AS value,',
    '    COUNT(*) AS cnt,',
    '    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct',
    'FROM matches_raw',
    'GROUP BY "{col}"',
    'ORDER BY cnt DESC',
    'LIMIT 30',
    "```",
    "",
    "### name (cardinality only)",
    "```sql",
    name_sql.strip(),
    "```",
    "",
    "### colorHex (cardinality only)",
    "```sql",
    colorhex_sql.strip(),
    "```",
    "",
]

In [91]:
md_lines += [
    "## E. Boolean field census",
    "",
    "```sql",
    "-- Per boolean column:",
    "SELECT",
    "    '{col}' AS column_name,",
    '    COUNT(*) FILTER (WHERE "{col}" = TRUE) AS true_count,',
    '    COUNT(*) FILTER (WHERE "{col}" = FALSE) AS false_count,',
    '    COUNT(*) - COUNT("{col}") AS null_count',
    "FROM matches_raw",
    "```",
    "",
    "## F. Numeric descriptive statistics",
    "",
    "```sql",
    "-- Per numeric column (matches_raw, ratings_raw, leaderboards_raw):",
    "SELECT",
    "    '{col}' AS column_name,",
    '    MIN("{col}") AS min_val,',
    '    MAX("{col}") AS max_val,',
    '    ROUND(AVG("{col}"), 2) AS mean_val,',
    '    ROUND(MEDIAN("{col}"), 2) AS median_val,',
    '    ROUND(STDDEV("{col}"), 2) AS stddev_val,',
    "    PERCENTILE_CONT(0.05) WITHIN GROUP"
    ' (ORDER BY "{col}") AS p05,',
    "    PERCENTILE_CONT(0.25) WITHIN GROUP"
    ' (ORDER BY "{col}") AS p25,',
    "    PERCENTILE_CONT(0.75) WITHIN GROUP"
    ' (ORDER BY "{col}") AS p75,',
    "    PERCENTILE_CONT(0.95) WITHIN GROUP"
    ' (ORDER BY "{col}") AS p95',
    "FROM <table>",
    'WHERE "{col}" IS NOT NULL',
    "```",
    "",
    "## F2. Zero counts for numeric columns",
    "",
    "`profiles_raw` excluded -- its only numeric column (`profileId`) is an identifier.",
    "",
    "```sql",
    "-- Per column (col in numeric zero cols list, table in matches_raw / ratings_raw / leaderboards_raw):",
    "SELECT",
    "    '{col}' AS column_name,",
    "    COUNT(*) FILTER (WHERE \"{col}\" = 0)                                                  AS zero_count,",
    "    COUNT(*) FILTER (WHERE \"{col}\" IS NOT NULL)                                          AS non_null_count,",
    "    ROUND(100.0 * COUNT(*) FILTER (WHERE \"{col}\" = 0)",
    "        / NULLIF(COUNT(*) FILTER (WHERE \"{col}\" IS NOT NULL), 0), 2)                    AS zero_pct_of_non_null",
    "FROM <table>",
    "```",
    "",
    "## F1b. Skewness and Kurtosis (EDA Manual Section 3.1)",
    "",
    "```sql",
    "-- Per numeric column (col in numeric cols list, table in matches_raw / ratings_raw / leaderboards_raw):",
    "SELECT",
    "    '{col}' AS column_name,",
    '    ROUND(SKEWNESS("{col}"), 4) AS skewness,',
    '    ROUND(KURTOSIS("{col}"), 4) AS kurtosis',
    "FROM <table>",
    'WHERE "{col}" IS NOT NULL',
    "```",
    "",
    "matches_raw numeric columns (9): rating, ratingDiff, population, slot, color,",
    "    team, speedFactor, treatyLength, internalLeaderboardId",
    "ratings_raw numeric columns (5): leaderboard_id, season, rating, games, rating_diff",
    "leaderboards_raw numeric columns (10): rank, rating, wins, losses, games,",
    "    streak, drops, rankCountry, season, rankLevel",
    "",
    "## G. Temporal range",
    "",
    "### matches_raw temporal range",
    "```sql",
    temporal_range_sql.strip(),
    "```",
    "",
    "### ratings_raw temporal range",
    "```sql",
    ratings_temporal_sql.strip(),
    "```",
    "",
    "### Match duration distribution",
    "```sql",
    duration_sql.strip(),
    "```",
    "",
    "### Excluded rows",
    "```sql",
    excluded_sql.strip(),
    "```",
    "",
    "### Monthly match counts",
    "```sql",
    monthly_sql.strip(),
    "```",
    "",
]

In [92]:
md_lines += [
    "## H. Auxiliary table NULL census",
    "",
    "```sql",
    "SUMMARIZE leaderboards_raw",
    "SUMMARIZE profiles_raw",
    "SUMMARIZE ratings_raw",
    "SELECT COUNT(*) FROM leaderboards_raw",
    "SELECT COUNT(*) FROM profiles_raw",
    "SELECT COUNT(*) FROM ratings_raw",
    "```",
    "",
    "### H.1b leaderboards_raw categorical, boolean, and temporal",
    "",
    "#### leaderboard VARCHAR (all values)",
    "```sql",
    lb_leaderboard_sql.strip(),
    "```",
    "",
    "#### country VARCHAR (top 30)",
    "```sql",
    lb_country_sql.strip(),
    "```",
    "",
    "#### active BOOLEAN census",
    "```sql",
    lb_boolean_sql.strip(),
    "```",
    "",
    "#### lastMatchTime and updatedAt temporal ranges",
    "```sql",
    lb_temporal_sql.strip(),
    "```",
    "",
    "#### ratings_raw leaderboard_id distribution",
    "```sql",
    rt_lb_id_sql.strip(),
    "```",
    "",
    "### H.2b profiles_raw categorical",
    "",
    "Note: 7 dead columns documented in constant_fields (sharedHistory, twitchChannel,",
    "youtubeChannel, youtubeChannelName, discordId, discordName, discordInvitation).",
    "",
    "#### country VARCHAR (top 30)",
    "```sql",
    pr_country_sql.strip(),
    "```",
    "",
    "#### clan VARCHAR (top 30)",
    "```sql",
    pr_clan_sql.strip(),
    "```",
    "",
    "## I. Dead/constant/near-constant field detection",
    "",
    "### I.1 Dead fields (BLOCKER fix)",
    "",
    "Detection uses (approx_cardinality <= 1) OR (null_pct >= 100.0) -- the OR condition",
    "catches profiles_raw columns that are 100% NULL but have phantom HyperLogLog",
    "cardinalities > 1 due to approximation error (Flajolet et al. 2007).",
    "",
    "### I.2 Near-constant fields (two-bucket split, EDA Manual Section 3.3)",
    "",
    "**Threshold:** NEAR_CONSTANT_CARDINALITY_THRESHOLD = 100",
    "**Rationale (I7):** max clearly categorical cardinality is ~68 (civ); next meaningful",
    "categorical is map (~261). At N=277M rows, uniqueness_ratio < 0.001 flags all columns",
    "with < 277,000 distinct values -- including semantically important categoricals.",
    "",
    "- **Bucket 1 (near_constant_low_cardinality):** uniqueness_ratio < 0.001 AND",
    "  approx_cardinality in [2, 100) AND null_pct < 100 -- genuinely low-cardinality",
    "- **Bucket 2 (near_constant_ratio_flagged):** uniqueness_ratio < 0.001 AND",
    "  approx_cardinality >= 100 -- ratio-flagged but NOT true near-constants",
    "  (includes map=261, which is semantically important)",
    "",
    "### I2. Duplicate row detection",
    "",
    "#### matches_raw (matchId, profileId) pairs",
    "```sql",
    dup_matches_sql.strip(),
    "```",
    "",
    "#### ratings_raw (profile_id, leaderboard_id, date) triples",
    "```sql",
    dup_ratings_sql.strip(),
    "```",
    "",
    "### I3. NULL co-occurrence for 0.15%-0.16% clusters",
    "",
    "Two distinct NULL clusters:",
    "- **Cluster A** (8 cols, null_count=415,649): allowCheats, lockSpeed, lockTeams,",
    "  recordGame, sharedExploration, teamPositions, teamTogether, turboMode",
    "- **Cluster B** (2 cols, null_count=443,358): fullTechTree, population",
    "",
    "#### Cluster A co-occurrence",
    "```sql",
    null_cooc_a_sql.strip(),
    "```",
    "",
    "#### Cluster B co-occurrence",
    "```sql",
    null_cooc_b_sql.strip(),
    "```",
    "",
    "#### Cross-cluster overlap",
    "```sql",
    null_cooc_cross_sql.strip(),
    "```",
    "",
    "### T07. Memory footprint",
    "",
    "DuckDB file size recorded via `os.path.getsize(str(db._dataset.db_file))`.",
    "",
    "## Post-game field annotations (Invariant #3)",
    "",
    "Columns encoding match outcome -- temporal leakage risk if used as features.",
    "",
    "| table | column | type | reason |",
    "|-------|--------|------|--------|",
] + [
    f"| {f['table']} | {f['column']} | {f['type']} | {f['reason']} |"
    for f in findings["post_game_fields"]
] + [
    "",
    "## Field Classification (preliminary)",
    "",
    f"Table: matches_raw | Status: preliminary | Columns annotated: {len(findings['field_classification']['fields'])}",
    "",
    "Formal boundary deferred to Phase 02 (Feature Engineering).",
    "",
    "| column | temporal_class | notes |",
    "|--------|---------------|-------|",
] + [
    f"| {f['column']} | {f['temporal_class']} | {f['notes']} |"
    for f in findings["field_classification"]["fields"]
] + [
    "",
    "---",
    "",
    "*Generated by notebook "
    "01_02_04_univariate_census.py*",
]

md_text = "\n".join(md_lines)
md_path = artifacts_dir / "01_02_04_univariate_census.md"
Path(md_path).write_text(md_text)
print(f"Markdown artifact saved: {md_path}")

Markdown artifact saved: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/02_eda/01_02_04_univariate_census.md


In [93]:
# Close DB connection
db.close()
print("Done. All artifacts written.")

Done. All artifacts written.
